# Phase 1 — SDT Unfrozen Baseline
## Cell 1: Environment Setup

This notebook implements the **unfrozen backbone** version of the SDT token pruning baseline.
Both the DiT backbone and the SDT gate are trained jointly using differential learning rates.

**This is a dry run (10 iterations).** All hyperparameters are set for fast testing on Colab.
When moving to RunPod, only change the values marked with `# RUNPOD: change this`.

### What this cell does
- Installs required packages
- Clones the DiT repository
- Imports all libraries
- Detects and prints GPU information

In [1]:
# ── Installs ───────────────────────────────────────────────────────────────────
# !pip install timm diffusers accelerate datasets fvcore -q

# ── DiT repo ───────────────────────────────────────────────────────────────────
import os, sys

if not os.path.exists('DiT'):
    !git clone https://github.com/facebookresearch/DiT.git -q
    print("DiT repo cloned.")
else:
    print("DiT repo already exists.")

sys.path.insert(0, os.path.abspath('DiT'))

# ── Imports ────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from diffusers.models import AutoencoderKL
from torchvision import transforms
from datasets import load_dataset
from tqdm import tqdm
from models import DiT_models

# ── Device ─────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── GPU info ───────────────────────────────────────────────────────────────────
print("=" * 52)
print(f"  Device         : {device}")
if device == "cuda":
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_ver  = torch.version.cuda
    print(f"  GPU            : {gpu_name}")
    print(f"  VRAM           : {vram_gb:.1f} GB")
    print(f"  CUDA version   : {cuda_ver}")
    print(f"  PyTorch version: {torch.__version__}")
else:
    print("  WARNING: No GPU detected. Training will be very slow.")
print("=" * 52)

# ── Dry run flag ───────────────────────────────────────────────────────────────
DRY_RUN = False   # RUNPOD: set to False for full training run
if DRY_RUN:
    print("\n  [DRY RUN MODE] — 10 iterations only.")
    print("  Set DRY_RUN = False when running on RunPod.\n")


# ── Blackwell optimisations ────────────────────────────────────────────────────
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')   # enables TF32 globally
print("Blackwell optimisations enabled: TF32, cudnn benchmark")

DiT repo cloned.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


  Device         : cuda
  GPU            : Tesla T4
  VRAM           : 15.6 GB
  CUDA version   : 12.8
  PyTorch version: 2.11.0+cu128
Blackwell optimisations enabled: TF32, cudnn benchmark


In [2]:
torch.backends.cudnn.benchmark    = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32   = True

## Cell 2: Model Loading

Loads DiT-XL/2 pretrained checkpoint and VAE.

**Key difference from frozen notebook:** backbone parameters are NOT frozen.
Both the DiT backbone and the SDT gate will update during training — but at
different learning rates (set in Cell 6).

### Why we do not freeze here
When the backbone is frozen, the gate learns to work around a model that was
never designed to have tokens skipped. The backbone does not cooperate.
With unfrozen joint training, the backbone adapts alongside the gate —
co-adapting to produce features that make token selection easier and more accurate.

### Memory implication
Unfrozen backbone means gradients flow through all 675M parameters.
This requires ~8GB more VRAM than the frozen version.
Gradient checkpointing (Cell 7) recovers most of this.

## Cell 3: Latent Cache

Builds or loads a cache of pre-encoded VAE latents for training.

**Dry run / Colab:** uses `zh-plus/tiny-imagenet` — public, no authentication,
no manual download. 512 latents, builds in ~2 minutes.

**RunPod full run:** replace with ImageNet-256 latents.
All final thesis metrics must use real ImageNet-256.
A flag `USE_TINY_IMAGENET` controls which path runs.

### Why we cache latents
Encoding images through the VAE every training step wastes GPU time.
The VAE is frozen so its output never changes for the same image.
Pre-encoding once and caching saves roughly 30% of training time.

### Latent shape
Each image encodes to `[4, 32, 32]` — 4 channels at 32×32 spatial resolution.
DiT operates entirely in this latent space, never in pixel space.

In [3]:
# ── DiT-XL/2 checkpoint ────────────────────────────────────────────────────────
CKPT = "DiT-XL-2-256x256.pt"

if not os.path.exists(CKPT):
    print("Downloading DiT-XL/2 checkpoint (~2.7GB)...")
    !wget -q --show-progress https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt
    print("Download complete.")
else:
    print(f"Checkpoint found: {CKPT}")

# ── Load DiT ───────────────────────────────────────────────────────────────────
print("\nLoading DiT-XL/2...")
dit = DiT_models["DiT-XL/2"](input_size=32).to(device)
state = torch.load(CKPT, map_location=device)
dit.load_state_dict(state.get("model", state))
dit.train()   # train mode — backbone updates during training

# ── Do NOT freeze backbone ─────────────────────────────────────────────────────
# Unlike the frozen notebook, we leave all parameters requiring grad = True
# The differential learning rate optimizer (Cell 6) handles the update speed
for p in dit.parameters():
    p.requires_grad = False

# ── Load VAE ───────────────────────────────────────────────────────────────────
print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse"
).to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False   # VAE always frozen — we never update it

# ── Architecture constants ─────────────────────────────────────────────────────
hidden_dim   = dit.x_embedder.proj.out_channels   # 1152 for DiT-XL
IN_CHANNELS  = 4                                   # VAE latent channels

# ── Parameter count breakdown ──────────────────────────────────────────────────
backbone_params = sum(p.numel() for p in dit.parameters())
vae_params      = sum(p.numel() for p in vae.parameters())

print("\n" + "=" * 52)
print(f"  DiT-XL/2 hidden dim  : {hidden_dim}")
print(f"  DiT backbone params  : {backbone_params/1e6:.1f}M")
print(f"  VAE params (frozen)  : {vae_params/1e6:.1f}M")
print(f"  Backbone requires_grad: "
      f"{all(p.requires_grad for p in dit.parameters())}")
print(f"  VAE requires_grad    : "
      f"{any(p.requires_grad for p in vae.parameters())}")
print("=" * 52)
print("\nModel loading complete.")

DiT-XL-2-256x256.pt 100%[===================>]   2.51G   160MB/s    in 15s     
Download complete.

Loading DiT-XL/2...
Loading VAE...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]


  DiT-XL/2 hidden dim  : 1152
  DiT backbone params  : 675.1M
  VAE params (frozen)  : 83.7M
  Backbone requires_grad: False
  VAE requires_grad    : False

Model loading complete.


In [4]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

ENCODE_BATCH      = 128
N_CACHE           = 9000
LATENT_CACHE_FILE = "imagenette_latent_cache.pt"

# ── load if exists, build if not ───────────────────────────────────────────────
if os.path.exists(LATENT_CACHE_FILE):
    print(f"Cache found — loading from {LATENT_CACHE_FILE}...")
    cache        = torch.load(LATENT_CACHE_FILE, map_location='cpu')
    latent_cache = cache["latents"]
    label_cache  = cache["labels"]
    print(f"Loaded {latent_cache.shape[0]} latents — shape {latent_cache.shape}")

else:
    print(f"No cache found — building from ImageNette ({N_CACHE} images)...")

    if not os.path.exists('/workspace/imagenette2-320'):
        os.system('wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -P /workspace/')
        os.system('tar -xf /workspace/imagenette2-320.tgz -C /workspace/')
        print("ImageNette extracted.")

    tf = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])

    dataset    = ImageFolder("/workspace/imagenette2-320/train", transform=tf)
    dataloader = DataLoader(
        dataset,
        batch_size    = ENCODE_BATCH,
        shuffle       = True,
        num_workers   = 8,
        pin_memory    = True,
        prefetch_factor = 2
    )

    all_latents, all_labels = [], []

    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        with torch.no_grad():
            lat = vae.encode(imgs).latent_dist.sample() * 0.18215
        all_latents.append(lat.cpu())
        all_labels.extend(labels.tolist())
        print(f"  {len(all_labels)}/{N_CACHE}", end='\r')
        if len(all_labels) >= N_CACHE:
            break

    latent_cache = torch.cat(all_latents)[:N_CACHE]
    label_cache  = torch.tensor(all_labels[:N_CACHE])

    torch.save(
        {"latents": latent_cache, "labels": label_cache},
        LATENT_CACHE_FILE
    )
    print(f"\nCached {latent_cache.shape[0]} latents → {LATENT_CACHE_FILE}")

# ── verify ─────────────────────────────────────────────────────────────────────
print(f"latent_cache : {latent_cache.shape}  dtype={latent_cache.dtype}")
print(f"label_cache  : {label_cache.shape}   range={label_cache.min().item()}-{label_cache.max().item()}")

No cache found — building from ImageNette (9000 images)...
ImageNette extracted.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 599.81 MiB is free. Including non-PyTorch memory, this process has 13.97 GiB memory in use. Of the allocated memory 13.49 GiB is allocated by PyTorch, and 374.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Cell 4: Core Architecture + Hook Manager

Defines all SDT token pruning components and a persistent hook management system.

### Components
- `modulate` — adaLN scale and shift helper
- `_run_attn_half` — MHSA sub-block, returns post-attention features + MLP params
- `_run_mlp_half` — MLP sub-block on token subset
- `TokenPredictor` — SDT gate (2-layer MLP + Gumbel-Softmax)
- `PruningWrapper` — wraps DiT block, applies gate between MHSA and MLP
- `set_pruning_mode` / `get_all_masks` — training utilities
- `HookManager` — persistent diagnostic hooks, zero cost when inactive

### Hook design philosophy
Hooks are registered **once** at startup and never removed.
Each hook group is activated/deactivated via a boolean flag.
When inactive a hook does nothing — one boolean check, no overhead.
This eliminates all stale hook errors and makes future extensions trivial.

### Available hook groups
| Key | What it captures | Used for |
|---|---|---|
| `gradient` | Token embedding gradients | Gradient saliency maps |
| `attention` | Per-layer attention entropy | Attention saliency maps |
| `token_feat` | Per-layer token features | Future: trajectory analysis |
| `gate_logit` | Raw gate logit distributions | Future: MAPPO state design |

In [ ]:
from diffusion import create_diffusion

# ── adaLN helper ───────────────────────────────────────────────────────────────
def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


# ── Attention half ─────────────────────────────────────────────────────────────
def _run_attn_half(block, x, c):
    """
    Run MHSA sub-block only. Full MHSA — all heads always active.
    Computes adaLN modulation once and returns MLP params for reuse.
    Returns: (x_post_attn [B,N,D], mlp_params tuple)
    """
    shift_msa, scale_msa, gate_msa, \
    shift_mlp, scale_mlp, gate_mlp = \
        block.adaLN_modulation(c).chunk(6, dim=1)

    attn_out    = block.attn(modulate(block.norm1(x), shift_msa, scale_msa))
    x_post_attn = x + gate_msa.unsqueeze(1) * attn_out
    mlp_params  = (shift_mlp, scale_mlp, gate_mlp)
    return x_post_attn, mlp_params


# ── MLP half ───────────────────────────────────────────────────────────────────
def _run_mlp_half(block, x, mlp_params):
    """
    Run MLP sub-block only on token subset x.
    x can be any subset — MLP has no cross-token interaction.
    Returns: x + residual [same shape as x]
    """
    shift_mlp, scale_mlp, gate_mlp = mlp_params
    mlp_in  = modulate(block.norm2(x), shift_mlp, scale_mlp)
    mlp_out = block.mlp(mlp_in)
    return x + gate_mlp.unsqueeze(1) * mlp_out


# ── SDT gate ───────────────────────────────────────────────────────────────────
class TokenPredictor(nn.Module):
    """
    SDT router: post-MHSA features → binary keep/skip mask per token.
    Architecture: LayerNorm → Linear(D→64) → GELU → Linear(64→1) → Gumbel-Softmax.

    This is the component MAPPO replaces in Phase 2.
    The Gumbel-Softmax straight-through estimator allows gradients to flow
    through the binary decision during training.
    """
    def __init__(self, d_model: int, bottleneck: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, bottleneck),
            nn.GELU(),
            nn.Linear(bottleneck, 1),
        )

    def forward(self, x: torch.Tensor, temperature: float = 1.0):
        """
        x          : [B, N, D] post-MHSA token features
        temperature: Gumbel-Softmax temperature, anneals 2.0→0.5 during training
        returns    : keep_mask [B, N] ∈ {0,1}, logits [B, N, 1]
        """
        logits    = self.mlp(x)
        pair      = torch.cat([logits, -logits], dim=-1)
        hard      = F.gumbel_softmax(pair, tau=temperature, hard=True)
        keep_mask = hard[..., 1]
        return keep_mask, logits


# ── PruningWrapper ─────────────────────────────────────────────────────────────
class PruningWrapper(nn.Module):
    """
    Wraps a single DiT block and intercepts its forward pass.

    Forward flow:
        x → MHSA (all tokens, all heads)
          → gate (TokenPredictor on post-MHSA features)
          → gather kept tokens
          → MLP (kept tokens only)
          → scatter back
          → output

    _live_mask : grad_fn intact, used for sparsity loss
    _last_mask : detached, used for logging and saliency
    """
    def __init__(self, block: nn.Module, predictor: TokenPredictor):
        super().__init__()
        self.block           = block
        self.predictor       = predictor
        self.pruning_enabled = True
        self._last_mask      = None
        self._live_mask      = None
        self._temperature    = 1.0

    def forward(self, x: torch.Tensor, c: torch.Tensor):
        if not self.pruning_enabled:
            self._live_mask = None
            return self.block(x, c)

        B, N, D = x.shape

        # step 1: full MHSA
        x_post_attn, mlp_params = _run_attn_half(self.block, x, c)

        # step 2: gate decision on post-MHSA features
        keep_mask, logits = self.predictor(x_post_attn, self._temperature)
        self._live_mask   = keep_mask
        self._last_mask   = keep_mask.detach()

        # step 3: per-image gather → MLP → scatter
        out = torch.zeros_like(x_post_attn)

        for b in range(B):
            keep_idx_b = keep_mask[b].detach().bool()
            N_kept     = keep_idx_b.sum().item()

            mlp_params_b = (
                mlp_params[0][b].unsqueeze(0),
                mlp_params[1][b].unsqueeze(0),
                mlp_params[2][b].unsqueeze(0),
            )

            if N_kept == 0 or N_kept == N:
                out[b] = _run_mlp_half(
                    self.block,
                    x_post_attn[b].unsqueeze(0),
                    mlp_params_b
                ).squeeze(0)
                continue

            idx_b    = keep_idx_b.nonzero(as_tuple=True)[0]
            x_kept   = x_post_attn[b, idx_b, :].unsqueeze(0)
            out_kept = _run_mlp_half(
                self.block, x_kept, mlp_params_b
            ).squeeze(0)

            out_b        = x_post_attn[b].clone()
            out_b[idx_b] = out_kept
            out[b]       = out_b

        return out


# ── Utilities ──────────────────────────────────────────────────────────────────
def set_pruning_mode(model: nn.Module, enabled: bool):
    for block in model.blocks:
        if isinstance(block, PruningWrapper):
            block.pruning_enabled = enabled


def get_all_masks(model: nn.Module, live: bool = False):
    attr = '_live_mask' if live else '_last_mask'
    return [
        getattr(b, attr) for b in model.blocks
        if isinstance(b, PruningWrapper) and getattr(b, attr) is not None
    ]


# ── Hook Manager ───────────────────────────────────────────────────────────────
class HookManager:
    """
    Persistent diagnostic hooks registered once, never removed.
    Zero overhead when inactive — one boolean check per hook call.

    Usage:
        hook_manager.activate('gradient')
        hook_manager.clear('gradient')
        _ = model(z_t, t, y)
        data = hook_manager.get('gradient')
        hook_manager.deactivate('gradient')
    """

    def __init__(self):
        self.active = {
            'gradient'  : False,   # token embedding gradients
            'attention' : False,   # per-layer attention entropy
            'token_feat': False,   # per-layer token features (future)
            'gate_logit': False,   # raw gate logit distributions (future)
        }
        self._handles  = []
        self._storage  = {k: [] for k in self.active}
        self._registered = False

    # ── activation controls ────────────────────────────────────────────────────
    def activate(self, *keys):
        for k in keys:
            if k not in self.active:
                print(f"  Warning: unknown hook key '{k}'")
                continue
            self.active[k] = True

    def deactivate(self, *keys):
        for k in keys:
            if k in self.active:
                self.active[k] = False

    def clear(self, key=None):
        if key:
            self._storage[key] = []
        else:
            for k in self._storage:
                self._storage[k] = []

    def get(self, key):
        return self._storage[key]

    def status(self):
        print("HookManager status:")
        for k, v in self.active.items():
            state = "ON " if v else "off"
            n     = len(self._storage[k])
            print(f"  {k:12s} : {state}  |  {n} items stored")

    # ── hook registration (call once after wrapping layers) ───────────────────
    def register_all(self, model):
        if self._registered:
            print("Hooks already registered — skipping.")
            return

        # ── gradient saliency: hook on x_embedder output ──────────────────────
        def embed_hook(module, inp, out):
            if not self.active['gradient']:
                return None
            out_grad = out.detach().requires_grad_(True)
            self._storage['gradient'].append(out_grad)
            return out_grad

        h = model.x_embedder.register_forward_hook(embed_hook)
        self._handles.append(h)

        # ── attention entropy: hook on every attn block ────────────────────────
        for block in model.blocks:
            b = block.block if isinstance(block, PruningWrapper) else block

            def make_attn_hook(storage, active_ref):
                def hook(module, inp, out):
                    if not active_ref['attention']:
                        return None
                    x       = inp[0]
                    B, N, D = x.shape
                    qkv     = module.qkv(x)
                    qkv     = qkv.reshape(
                        B, N, 3, module.num_heads,
                        D // module.num_heads
                    ).permute(2, 0, 3, 1, 4)
                    q, k, _ = qkv.unbind(0)
                    scale   = q.shape[-1] ** -0.5
                    attn    = (q @ k.transpose(-2, -1)) * scale
                    attn    = attn.softmax(dim=-1)
                    eps     = 1e-8
                    ent     = -(attn * (attn + eps).log()).sum(dim=-1)
                    ent     = ent.mean(dim=1)
                    storage.append(ent[0].detach().cpu())
                return hook

            h = b.attn.register_forward_hook(
                make_attn_hook(self._storage['attention'], self.active)
            )
            self._handles.append(h)

        # ── token features: hook on every block output (future use) ───────────
        for block in model.blocks:
            b = block.block if isinstance(block, PruningWrapper) else block

            def make_feat_hook(storage, active_ref):
                def hook(module, inp, out):
                    if not active_ref['token_feat']:
                        return None
                    storage.append(out.detach().cpu())
                return hook

            h = b.register_forward_hook(
                make_feat_hook(self._storage['token_feat'], self.active)
            )
            self._handles.append(h)

        # ── gate logits: hook on every TokenPredictor (future use) ────────────
        for block in model.blocks:
            if not isinstance(block, PruningWrapper):
                continue

            def make_gate_hook(storage, active_ref):
                def hook(module, inp, out):
                    if not active_ref['gate_logit']:
                        return None
                    _, logits = out
                    storage.append(logits.detach().cpu())
                return hook

            h = block.predictor.register_forward_hook(
                make_gate_hook(self._storage['gate_logit'], self.active)
            )
            self._handles.append(h)

        self._registered = True
        print(f"HookManager: {len(self._handles)} hooks registered across model.")
        print(f"All groups inactive by default. Call activate() to use.")


# ── instantiate ────────────────────────────────────────────────────────────────
hook_manager = HookManager()

print("\nCell 4 complete:")
print("  modulate           ✓")
print("  _run_attn_half     ✓")
print("  _run_mlp_half      ✓")
print("  TokenPredictor     ✓")
print("  PruningWrapper     ✓")
print("  set_pruning_mode   ✓")
print("  get_all_masks      ✓")
print("  HookManager        ✓  (register_all called in Cell 5)")

## Cell 5: Wrap Layers + Register Hooks

Wraps every other DiT block with a `PruningWrapper` containing a `TokenPredictor` gate.
After wrapping, registers all diagnostic hooks via `hook_manager.register_all()`.

### Why every other layer
14 of 28 layers gated — a practical starting point.
Experiment 3 (per-layer saliency) showed middle layers have the clearest
spatial structure, making them the best candidates for token pruning.
Early layers (0, 2) are the weakest — consider removing them in Phase 2.

### Wrapping is safe to re-run
The peel loop removes any existing PruningWrapper before re-wrapping.
This prevents the double-wrapping bug from previous sessions.

### Hook registration
`hook_manager.register_all()` is called here — after wrapping — so the
gate_logit hooks correctly attach to the TokenPredictor instances.
Hooks are registered exactly once. Re-running this cell is safe.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
PRUNING_LAYERS = list(range(0, 28, 2))   # [0,2,4,...,26] — every other layer
BOTTLENECK     = 64                       # TokenPredictor hidden dim

# ── Step 1: peel any existing wrappers ────────────────────────────────────────
print("Peeling any existing PruningWrappers...")
for idx in PRUNING_LAYERS:
    block = dit.blocks[idx]
    depth = 0
    while isinstance(block, PruningWrapper):
        block = block.block
        depth += 1
    dit.blocks[idx] = block
    if depth > 0:
        print(f"  Layer {idx:2d}: peeled {depth} wrapper(s)")

# ── Step 2: wrap with fresh PruningWrapper ────────────────────────────────────
print("\nWrapping layers with SDT gate...")
for idx in PRUNING_LAYERS:
    predictor       = TokenPredictor(hidden_dim, bottleneck=BOTTLENECK).to(device)
    dit.blocks[idx] = PruningWrapper(dit.blocks[idx], predictor)

# ── Step 3: collect gate references ───────────────────────────────────────────
all_parasites = [
    b.predictor for b in dit.blocks
    if isinstance(b, PruningWrapper)
]

# ── Step 4: verify wrapping depth ─────────────────────────────────────────────
print("\nVerifying wrapping depth...")
depth_ok = True
for idx in PRUNING_LAYERS:
    outer = type(dit.blocks[idx]).__name__
    inner = type(dit.blocks[idx].block).__name__
    if outer != 'PruningWrapper' or inner != 'DiTBlock':
        print(f"  Layer {idx:2d}: ✗  outer={outer}, inner={inner}")
        depth_ok = False
    else:
        print(f"  Layer {idx:2d}: ✓  PruningWrapper → DiTBlock")

if depth_ok:
    print("\n  All layers wrapped correctly — no double-wrapping.")
else:
    print("\n  WARNING: wrapping issue detected. Re-run this cell.")

# ── Step 5: parameter count breakdown ─────────────────────────────────────────
gate_params     = sum(
    p.numel() for pp in all_parasites
    for p in pp.parameters()
)
backbone_params = sum(
    p.numel() for p in dit.parameters()
) - gate_params

print("\n" + "=" * 52)
print(f"  Pruning layers   : {PRUNING_LAYERS}")
print(f"  Gates installed  : {len(all_parasites)}")
print(f"  Gate params      : {gate_params/1e6:.2f}M")
print(f"  Backbone params  : {backbone_params/1e6:.1f}M")
print(f"  Gate / backbone  : {gate_params/backbone_params*100:.3f}%")
print("=" * 52)

# ── Step 6: register all hooks now that wrappers are in place ─────────────────
print("\nRegistering diagnostic hooks...")
hook_manager.register_all(dit)
hook_manager.status()

print("\nCell 5 complete.")

In [ ]:
compiled_count = 0
for idx in range(len(dit.blocks)):
    if not isinstance(dit.blocks[idx], PruningWrapper):
        dit.blocks[idx] = torch.compile(
            dit.blocks[idx],
            mode='default'
        )
        compiled_count += 1

print(f"torch.compile applied to {compiled_count} unwrapped DiTBlocks")
print(f"Skipped {len(PRUNING_LAYERS)} PruningWrapper blocks (dynamic control flow)")
print(f"Total blocks: {len(dit.blocks)}")

## Cell 6: Differential Learning Rate Optimizer

Sets up the Adam optimizer with two separate parameter groups:

| Group | Parameters | Learning rate | Reason |
|---|---|---|---|
| Backbone | All DiT blocks, embedders, final layer | 1e-5 | Slow — preserve pretrained knowledge |
| Gate | All TokenPredictor MLPs | 1e-4 | Fast — gate needs to learn from scratch |

### Why different learning rates matter
The backbone has 675M parameters carefully trained over millions of steps on ImageNet.
A high learning rate would destroy this knowledge in the first few hundred steps.
The gate starts from random initialisation and needs a higher rate to learn quickly.

Using the same learning rate for both either:
- Destroys the backbone if the rate is too high
- Starves the gate if the rate is too low

Differential learning rates let both learn at their natural pace simultaneously.

### The 10× ratio
Backbone LR = 1e-5, gate LR = 1e-4 — a 10× difference.
This is a standard ratio for fine-tuning pretrained models with new components.
DyDiT uses a similar setup in their joint fine-tuning.

In [ ]:
from diffusion import create_diffusion

# ── Hyperparameters ────────────────────────────────────────────────────────────
# Training scale
STEPS        = 10     if DRY_RUN else 50000   # RUNPOD: 50000
BATCH_SIZE   = 2      if DRY_RUN else 32         # RUNPOD: 32
WARMUP_STEPS = 2      if DRY_RUN else 1000    # RUNPOD: 1000

# Learning rates
LR_BACKBONE  = 1e-5   # slow — preserve pretrained weights
LR_GATE      = 1e-4   # fast — gate learns from scratch

# Loss weights
LAMBDA_DIT    = 1.0   # weight on diffusion quality loss
LAMBDA_SPARSE = 0.5   # weight on token budget loss
LAMBDA_REG    = 0.1   # weight on backbone regularisation loss

# Token budget
TARGET_RATIO  = 0.5   # keep 50% of tokens on average

# Gumbel-Softmax temperature annealing
TEMP_START    = 2.0   # soft decisions early — easier to learn
TEMP_END      = 0.5   # sharper decisions by end of training

# Diffusion
IN_CHANNELS   = 4     # VAE latent channels (noise is always 4ch)

# ── Separate parameter groups ──────────────────────────────────────────────────
# Gate parameter IDs — used to exclude them from backbone group
gate_param_ids = {
    id(p)
    for pp in all_parasites
    for p in pp.parameters()
}

# Backbone: everything in DiT that is NOT a gate parameter
backbone_params_list = [
    p for p in dit.parameters()
    if id(p) not in gate_param_ids and p.requires_grad
]

# Gate: only the TokenPredictor parameters
gate_params_list = [
    p for pp in all_parasites
    for p in pp.parameters()
]

# ── Build optimizer with two groups ───────────────────────────────────────────
optimizer = torch.optim.Adam([
    # {
    #     'params' : backbone_params_list,
    #     'lr'     : LR_BACKBONE,
    #     'name'   : 'backbone'
    # },
    {
        'params' : gate_params_list,
        'lr'     : LR_GATE,
        'name'   : 'gate'
    },
])
from torch.optim.lr_scheduler import CosineAnnealingLR

scheduler = CosineAnnealingLR(
    optimizer,
    T_max   = STEPS,
    eta_min = 1e-7
)
print(f"Scheduler: CosineAnnealingLR, T_max={STEPS}, eta_min=1e-7")
# ── Diffusion schedule ─────────────────────────────────────────────────────────
diffusion = create_diffusion(timestep_respacing="")

# ── History dict ───────────────────────────────────────────────────────────────
history = {
    'loss_dit'     : [],
    'loss_sparse'  : [],
    'loss_reg'     : [],
    'loss_total'   : [],
    'loss_teacher' : [],
    'keep_ratio'   : [],
    'lr_backbone'  : [],
    'lr_gate'      : [],
    'temperature'  : [],
}

# ── Verify optimizer groups ────────────────────────────────────────────────────
total_opt_params = sum(
    len(g['params']) for g in optimizer.param_groups
)
backbone_count = len(backbone_params_list)
gate_count     = len(gate_params_list)

print("=" * 52)
print(f"  Optimizer        : Adam")
print(f"  Backbone params  : {backbone_count:,}  lr={LR_BACKBONE}")
print(f"  Gate params      : {gate_count:,}  lr={LR_GATE}")
print(f"  Total opt params : {total_opt_params:,}")
print(f"  STEPS            : {STEPS}")
print(f"  BATCH_SIZE       : {BATCH_SIZE}")
print(f"  WARMUP_STEPS     : {WARMUP_STEPS}")
print(f"  LAMBDA_DIT       : {LAMBDA_DIT}")
print(f"  LAMBDA_SPARSE    : {LAMBDA_SPARSE}")
print(f"  LAMBDA_REG       : {LAMBDA_REG}")
print(f"  TARGET_RATIO     : {TARGET_RATIO}")
print(f"  TEMP_START→END   : {TEMP_START} → {TEMP_END}")
print("=" * 52)

if DRY_RUN:
    print(f"\n  [DRY RUN] STEPS={STEPS}, BATCH_SIZE={BATCH_SIZE}")
    print(f"  Change DRY_RUN=False for full RunPod training.\n")

## Cell 7: Gradient Checkpointing

Enables gradient checkpointing on the DiT backbone to reduce VRAM usage.

### The memory problem
Unfrozen training stores gradients for all 675M backbone parameters.
Without checkpointing, PyTorch saves every intermediate activation
at every layer during the forward pass — needed for backpropagation.
For DiT-XL at batch size 32 this requires ~20GB of activation memory alone.

### What gradient checkpointing does
Instead of saving all activations, it saves only at checkpoint boundaries
and recomputes the rest during the backward pass on demand.

| | Normal | Checkpointing |
|---|---|---|
| Memory | High | ~40% lower |
| Speed | Baseline | ~30% slower backward |
| Result | OOM on A4500 | Runs comfortably |

### When to use
| GPU | VRAM | Use checkpointing? |
|---|---|---|
| A4500 (yours) | 20GB | Yes — required |
| A6000 48GB | 48GB | Optional |
| H100 SXM | 80GB | Not needed |

The flag `USE_GRAD_CHECKPOINT` controls this.
Set False on H100 for faster training.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
USE_GRAD_CHECKPOINT = True   # RUNPOD H100: set False — not needed at 80GB VRAM

# ── Enable gradient checkpointing ─────────────────────────────────────────────
if USE_GRAD_CHECKPOINT:
    # PyTorch's built-in activation checkpointing
    # Applied to each DiT transformer block individually
    # Blocks that are PruningWrappers checkpoint the inner DiTBlock
    from torch.utils.checkpoint import checkpoint

    # Store original forward methods so we can restore if needed
    _original_forwards = {}

    for idx, block in enumerate(dit.blocks):
        if isinstance(block, PruningWrapper):
            # checkpoint the inner DiTBlock forward
            inner = block.block
            _original_forwards[idx] = inner.forward

            def make_checkpointed_forward(inner_block):
                def checkpointed_forward(x, c):
                    def run(x, c):
                        return inner_block.__class__.forward(inner_block, x, c)
                    return checkpoint(run, x, c, use_reentrant=False)
                return checkpointed_forward

            inner.forward = make_checkpointed_forward(inner)

        else:
            # checkpoint unwrapped blocks directly
            _original_forwards[idx] = block.forward

            def make_checkpointed_forward_plain(b):
                def checkpointed_forward(x, c):
                    def run(x, c):
                        return b.__class__.forward(b, x, c)
                    return checkpoint(run, x, c, use_reentrant=False)
                return checkpointed_forward

            block.forward = make_checkpointed_forward_plain(block)

    print("Gradient checkpointing: ENABLED")
    print(f"  Applied to all {len(dit.blocks)} DiT blocks")
    print(f"  use_reentrant=False — compatible with torch.compile")

else:
    print("Gradient checkpointing: DISABLED")
    print(f"  Running on high-VRAM GPU — full activations stored")

# ── VRAM estimate ──────────────────────────────────────────────────────────────
if device == "cuda":
    free_vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
    model_vram = sum(
        p.numel() * p.element_size()
        for p in dit.parameters()
    ) / 1e9

    print("\n" + "=" * 52)
    print(f"  Total VRAM available  : {free_vram:.1f} GB")
    print(f"  Model weights         : {model_vram:.2f} GB")
    print(f"  Estimated optimizer   : {model_vram * 2:.2f} GB")
    print(f"  Estimated activations : {'~2GB (checkpointed)' if USE_GRAD_CHECKPOINT else '~8GB (full)'}")
    print(f"  Estimated total       : {model_vram * 3 + (2 if USE_GRAD_CHECKPOINT else 8):.1f} GB")
    print(f"  Gradient checkpoint   : {'ON  ✓' if USE_GRAD_CHECKPOINT else 'OFF'}")

    if free_vram < 24 and not USE_GRAD_CHECKPOINT:
        print(f"\n  ⚠ WARNING: <24GB VRAM with checkpointing disabled.")
        print(f"  Set USE_GRAD_CHECKPOINT=True to avoid OOM.")
    print("=" * 52)

# ── Restore function (call if checkpointing causes issues) ─────────────────────
def disable_grad_checkpointing():
    """Restore original forward methods if checkpointing causes issues."""
    for idx, block in enumerate(dit.blocks):
        if idx in _original_forwards:
            if isinstance(block, PruningWrapper):
                block.block.forward = _original_forwards[idx]
            else:
                block.forward = _original_forwards[idx]
    print("Gradient checkpointing disabled — original forwards restored.")

print("\nCell 7 complete.")
print("  Call disable_grad_checkpointing() if you hit issues.")

In [ ]:
print(torch.cuda.is_bf16_supported())
print(torch.cuda.get_device_capability())

## Cell 8: Training Loop

The main training loop for unfrozen joint backbone + gate training.

### Three loss terms

**L_DiT (quality loss)**
MSE between student noise prediction and true noise.
Drives the gate to skip only tokens that don't hurt prediction quality.
Gradient flows through both the gate and the backbone.

**L_sparse (budget loss)**
Squared deviation of average keep ratio from target (50%).
Forces the gate to actually skip tokens rather than keeping everything.
Without this loss the gate collapses to keeping all tokens.

**L_reg (regularisation loss)**
MSE between current backbone features and frozen reference features.
Prevents the backbone from drifting too far from pretrained weights.
Acts as a soft anchor — backbone can adapt but not catastrophically forget.

### Warmup phase
For the first `WARMUP_STEPS` steps the gate temperature stays high (2.0)
and only the gate parameters update. The backbone is temporarily frozen.
This gives the gate time to find a reasonable operating point before
the backbone starts co-adapting. Prevents early training instability.

### Dry run
10 steps on Colab. Verifies the full pipeline runs end to end.
All loss terms, all gradients, all logging — identical to RunPod.
Only the step count and batch size differ.

In [ ]:
# ── Cell 8: Training Loop (FROZEN backbone, optimized, auto-resume) ───────────

# ── dtype + device guard ────────────────────────────────────────────────────────
latent_cache = latent_cache.to(device=device, dtype=torch.bfloat16)
label_cache  = label_cache.to(device)
dit          = dit.to(device=device, dtype=torch.bfloat16)
print(f"dtype guard: model={next(dit.parameters()).dtype}, cache={latent_cache.dtype}")

# ── confirm backbone is frozen ──────────────────────────────────────────────────
backbone_frozen = all(not p.requires_grad for n, p in dit.named_parameters()
                      if 'predictor' not in n)
print(f"Backbone frozen: {backbone_frozen}")

# ── auto-resume config ──────────────────────────────────────────────────────────
CHECKPOINT_INTERVAL = 1000
CHECKPOINT_PATH     = '/workspace/checkpoints/dit_sdt_frozen_resume.pt'
os.makedirs('/workspace/checkpoints', exist_ok=True)

# ── intervals (no teacher pass needed every step) ──────────────────────────────
TEACHER_INTERVAL = 200    # teacher_gap logged every 200 steps only

# ── check for existing checkpoint ─────────────────────────────────────────────
resume_step = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"\nResume checkpoint found: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    dit.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    if 'scheduler_state' in ckpt and scheduler is not None:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    history          = ckpt['history']
    resume_step      = ckpt['step'] + 1
    last_teacher_gap = ckpt['last_teacher_gap']
    print(f"  Resuming from step {resume_step} / {STEPS}")
else:
    print(f"\nNo checkpoint found — starting from step 0")
    last_teacher_gap = 0.0
    history = {
        'loss_dit'     : [],
        'loss_sparse'  : [],
        'loss_total'   : [],
        'loss_teacher' : [],
        'keep_ratio'   : [],
        'lr_gate'      : [],
        'temperature'  : [],
    }

# ── training loop ──────────────────────────────────────────────────────────────
print(f"\nStarting FROZEN training — {STEPS} steps | batch={BATCH_SIZE} | bf16")
print(f"  Forward passes per step: 1 (student) + teacher every {TEACHER_INTERVAL}")
print("=" * 70)

dit.eval()                         # frozen backbone — eval mode throughout
for pp in all_parasites:
    pp.train()                     # gate predictors in train mode

pbar = tqdm(range(resume_step, STEPS))

for step in pbar:
    optimizer.zero_grad()

    # ── temperature annealing ──────────────────────────────────────────────────
    frac        = step / max(STEPS - 1, 1)
    temperature = TEMP_START * (TEMP_END / TEMP_START) ** frac
    for pp in all_parasites:
        pp._temperature = temperature

    # ── sample batch — cache on GPU, no transfer ──────────────────────────────
    idx_batch = torch.randint(0, latent_cache.shape[0], (BATCH_SIZE,))
    x0        = latent_cache[idx_batch]
    y         = label_cache[idx_batch]
    t         = torch.randint(0, 1000, (BATCH_SIZE,), device=device)
    noise     = torch.randn_like(x0)
    z_t       = diffusion.q_sample(x0, t, noise=noise).to(torch.bfloat16)

    # ── teacher pass: only every TEACHER_INTERVAL steps (diagnostic only) ─────
    if step % TEACHER_INTERVAL == 0:
        set_pruning_mode(dit, enabled=False)
        with torch.inference_mode():
            teacher_out        = dit(z_t, t, y)
            noise_pred_teacher = teacher_out[:, :IN_CHANNELS, :, :]

    # ── student pass: the only mandatory forward pass ─────────────────────────
    set_pruning_mode(dit, enabled=True)
    student_out        = dit(z_t, t, y)
    noise_pred_student = student_out[:, :IN_CHANNELS, :, :]
    loss_dit           = F.mse_loss(noise_pred_student, noise)

    if step % TEACHER_INTERVAL == 0:
        last_teacher_gap = F.mse_loss(
            noise_pred_student.detach(),
            noise_pred_teacher
        ).item()

    # ── sparsity loss ─────────────────────────────────────────────────────────
    live_masks  = get_all_masks(dit, live=True)
    live_ratio  = torch.stack(
        [m.float().mean() for m in live_masks]
    ).mean()
    loss_sparse = (live_ratio - TARGET_RATIO) ** 2

    # ── total loss + backward (gate only — backbone frozen) ───────────────────
    total_loss = (
        LAMBDA_DIT    * loss_dit +
        LAMBDA_SPARSE * loss_sparse
    )
    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        [p for pp in all_parasites for p in pp.parameters()],
        max_norm=1.0
    )
    optimizer.step()
    if scheduler is not None:
        scheduler.step()

    # ── logging ────────────────────────────────────────────────────────────────
    log_masks   = get_all_masks(dit, live=False)
    log_ratio   = torch.stack(
        [m.float().mean() for m in log_masks]
    ).mean().item()
    lr_gate     = optimizer.param_groups[0]['lr']

    history['loss_dit'].append(loss_dit.item())
    history['loss_sparse'].append(loss_sparse.item())
    history['loss_total'].append(total_loss.item())
    history['loss_teacher'].append(last_teacher_gap)
    history['keep_ratio'].append(log_ratio)
    history['lr_gate'].append(lr_gate)
    history['temperature'].append(temperature)

    pbar.set_description(
        f"dit={loss_dit.item():.4f} | "
        f"sp={loss_sparse.item():.4f} | "
        f"gap={last_teacher_gap:.4f} | "
        f"kept={log_ratio*100:.1f}% | "
        f"bs={BATCH_SIZE} | "
        f"t={temperature:.2f}"
    )

    # ── checkpoint every CHECKPOINT_INTERVAL steps ─────────────────────────────
    if step % CHECKPOINT_INTERVAL == 0 and step > resume_step:
        ckpt = {
            'step'             : step,
            'model_state'      : dit.state_dict(),
            'optimizer_state'  : optimizer.state_dict(),
            'scheduler_state'  : scheduler.state_dict() if scheduler is not None else None,
            'history'          : history,
            'last_teacher_gap' : last_teacher_gap,
            'pruning_layers'   : PRUNING_LAYERS,
            'hidden_dim'       : hidden_dim,
            'steps_total'      : STEPS,
            'batch_size'       : BATCH_SIZE,
            'target_ratio'     : TARGET_RATIO,
        }
        torch.save(ckpt, CHECKPOINT_PATH)
        pbar.write(f"  Checkpoint saved at step {step}")

print("\nTraining complete.")
print("=" * 70)
print(f"  Final loss_dit    : {history['loss_dit'][-1]:.4f}")
print(f"  Final loss_sparse : {history['loss_sparse'][-1]:.4f}")
print(f"  Final keep_ratio  : {history['keep_ratio'][-1]*100:.1f}%")
print(f"  Final teacher_gap : {history['loss_teacher'][-1]:.4f}")
print(f"  Steps trained     : {len(history['loss_dit'])}")

# ── final checkpoint ──────────────────────────────────────────────────────────
final_ckpt = {
    'step'             : STEPS - 1,
    'model_state'      : dit.state_dict(),
    'optimizer_state'  : optimizer.state_dict(),
    'history'          : history,
    'last_teacher_gap' : last_teacher_gap,
    'pruning_layers'   : PRUNING_LAYERS,
    'hidden_dim'       : hidden_dim,
    'steps_total'      : STEPS,
    'batch_size'       : BATCH_SIZE,
    'target_ratio'     : TARGET_RATIO,
}
torch.save(final_ckpt, '/workspace/checkpoints/dit_sdt_frozen_final.pt')
print(f"  Final checkpoint  : /workspace/checkpoints/dit_sdt_frozen_final.pt")

In [ ]:
# # ── Cell 8: Training Loop (unfrozen backbone, auto-resume) ────────────────────

# # ── dtype and device guard ─────────────────────────────────────────────────────
# latent_cache = latent_cache.to(device=device, dtype=torch.bfloat16)
# label_cache  = label_cache.to(device)
# dit          = dit.to(device=device, dtype=torch.bfloat16)

# print(f"dtype guard: model={next(dit.parameters()).dtype}, cache={latent_cache.dtype}")

# # ── auto-resume config ─────────────────────────────────────────────────────────
# CHECKPOINT_INTERVAL = 500    # save checkpoint every 500 steps
# CHECKPOINT_PATH     = '/workspace/checkpoints_1/dit_sdt_resume.pt'
# os.makedirs('/workspace/checkpoints_1', exist_ok=True)

# # ── check for existing checkpoint ─────────────────────────────────────────────
# resume_step = 0
# if os.path.exists(CHECKPOINT_PATH):
#     print(f"\nResume checkpoint found: {CHECKPOINT_PATH}")
#     ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

#     dit.load_state_dict(ckpt['model_state'])
#     optimizer.load_state_dict(ckpt['optimizer_state'])
#     scheduler.load_state_dict(ckpt['scheduler_state'])
#     history     = ckpt['history']
#     resume_step = ckpt['step'] + 1
#     last_loss_reg    = torch.tensor(ckpt['last_loss_reg'],    device=device)
#     last_teacher_gap = ckpt['last_teacher_gap']

#     print(f"  Resuming from step {resume_step} / {STEPS}")
#     print(f"  History length: {len(history['loss_dit'])} entries")
# else:
#     print(f"\nNo checkpoint found — starting from step 0")
#     last_loss_reg    = torch.tensor(0.0, device=device)
#     last_teacher_gap = 0.0
#     history = {
#         'loss_dit'     : [],
#         'loss_sparse'  : [],
#         'loss_reg'     : [],
#         'loss_total'   : [],
#         'loss_teacher' : [],
#         'keep_ratio'   : [],
#         'lr_backbone'  : [],
#         'lr_gate'      : [],
#         'temperature'  : [],
#     }

# # ── rebuild ref tensors after potential state load ─────────────────────────────
# # ref_z = latent_cache[:BATCH_SIZE]
# ref_z = latent_cache[:BATCH_SIZE].to(torch.bfloat16)
# ref_t = torch.zeros(BATCH_SIZE, dtype=torch.long, device=device)
# ref_y = label_cache[:BATCH_SIZE]

# # ── frozen reference snapshot ──────────────────────────────────────────────────
# print("\nBuilding regularisation reference snapshot...")

# set_pruning_mode(dit, enabled=False)
# dit.eval()

# ref_features = {}
# ref_hooks    = []

# def make_ref_hook(layer_idx):
#     def hook(module, inp, out):
#         ref_features[layer_idx] = out.detach()
#     return hook

# REF_LAYERS        = [0, 4, 8, 12, 16, 20, 24, 27]
# REG_INTERVAL      = 50
# TEACHER_INTERVAL  = 100

# for li in REF_LAYERS:
#     block = dit.blocks[li]
#     b     = block.block if isinstance(block, PruningWrapper) else block
#     h     = b.register_forward_hook(make_ref_hook(li))
#     ref_hooks.append(h)

# with torch.no_grad():
#     _ = dit(ref_z, ref_t, ref_y)

# for h in ref_hooks:
#     h.remove()

# dit.train()
# print(f"  Reference features captured at layers : {REF_LAYERS}")
# print(f"  Resuming from step                    : {resume_step}")
# print(f"  Remaining steps                       : {STEPS - resume_step}")
# print(f"  Checkpoint every                      : {CHECKPOINT_INTERVAL} steps")

# # ── training loop ──────────────────────────────────────────────────────────────
# print(f"\nStarting training — {STEPS} steps | batch={BATCH_SIZE} | {'[DRY RUN]' if DRY_RUN else '[FULL RUN]'}")
# print("=" * 70)

# pbar = tqdm(range(resume_step, STEPS))

# for step in pbar:
#     optimizer.zero_grad()

#     # ── temperature annealing ──────────────────────────────────────────────────
#     frac        = step / max(STEPS - 1, 1)
#     temperature = TEMP_START * (TEMP_END / TEMP_START) ** frac
#     for pp in all_parasites:
#         pp._temperature = temperature

#     # ── warmup ────────────────────────────────────────────────────────────────
#     in_warmup = step < WARMUP_STEPS
#     for p in backbone_params_list:
#         p.requires_grad = not in_warmup

#     # ── sample batch ───────────────────────────────────────────────────────────
#     idx_batch = torch.randint(0, latent_cache.shape[0], (BATCH_SIZE,))
#     x0        = latent_cache[idx_batch]
#     y         = label_cache[idx_batch]
#     t         = torch.randint(0, 1000, (BATCH_SIZE,), device=device)
#     noise     = torch.randn_like(x0)
#     z_t = diffusion.q_sample(x0, t, noise=noise)
#     z_t = z_t.to(torch.bfloat16)   # cast to match model dtype
#     # ── L_reg: separate backward every REG_INTERVAL steps ────────────────────
#     if not in_warmup and step % REG_INTERVAL == 0:
#         cur_features = {}
#         cur_hooks    = []

#         def make_cur_hook(layer_idx):
#             def hook(module, inp, out):
#                 cur_features[layer_idx] = out
#             return hook

#         set_pruning_mode(dit, enabled=False)
#         dit.train()

#         for li in REF_LAYERS:
#             block = dit.blocks[li]
#             b     = block.block if isinstance(block, PruningWrapper) else block
#             h     = b.register_forward_hook(make_cur_hook(li))
#             cur_hooks.append(h)

#         with torch.enable_grad():
#             _ = dit(ref_z, ref_t, ref_y)

#         for h in cur_hooks:
#             h.remove()

#         reg_terms = []
#         for li in REF_LAYERS:
#             if li not in cur_features:
#                 continue
#             abs_mse   = F.mse_loss(cur_features[li], ref_features[li])
#             ref_scale = ref_features[li].pow(2).mean() + 1e-8
#             reg_terms.append(abs_mse / ref_scale)

#         loss_reg = torch.stack(reg_terms).mean()
#         (LAMBDA_REG * loss_reg).backward()
#         last_loss_reg = loss_reg.detach()

#     # ── teacher pass: every TEACHER_INTERVAL steps only ───────────────────────
#     if step % TEACHER_INTERVAL == 0:
#         set_pruning_mode(dit, enabled=False)
#         dit.eval()
#         with torch.inference_mode():
#             teacher_out        = dit(z_t, t, y)
#             noise_pred_teacher = teacher_out[:, :IN_CHANNELS, :, :]

#     # ── student pass ──────────────────────────────────────────────────────────
#     set_pruning_mode(dit, enabled=True)
#     dit.train()
#     student_out        = dit(z_t, t, y)
#     noise_pred_student = student_out[:, :IN_CHANNELS, :, :]
#     loss_dit           = F.mse_loss(noise_pred_student, noise)

#     if step % TEACHER_INTERVAL == 0:
#         last_teacher_gap = F.mse_loss(
#             noise_pred_student.detach(),
#             noise_pred_teacher
#         ).item()

#     # ── L_sparse ──────────────────────────────────────────────────────────────
#     live_masks  = get_all_masks(dit, live=True)
#     live_ratio  = torch.stack(
#         [m.float().mean() for m in live_masks]
#     ).mean()
#     loss_sparse = (live_ratio - TARGET_RATIO) ** 2

#     # ── total loss ─────────────────────────────────────────────────────────────
#     total_loss = (
#         LAMBDA_DIT    * loss_dit +
#         LAMBDA_SPARSE * loss_sparse
#     )
#     total_loss.backward()

#     torch.nn.utils.clip_grad_norm_(backbone_params_list, max_norm=1.0)
#     torch.nn.utils.clip_grad_norm_(gate_params_list,     max_norm=1.0)
#     optimizer.step()
#     scheduler.step()

#     # ── logging ────────────────────────────────────────────────────────────────
#     log_masks   = get_all_masks(dit, live=False)
#     log_ratio   = torch.stack(
#         [m.float().mean() for m in log_masks]
#     ).mean().item()
#     reg_log     = last_loss_reg.item()
#     total_log   = total_loss.item() + LAMBDA_REG * reg_log

#     history['loss_dit'].append(loss_dit.item())
#     history['loss_sparse'].append(loss_sparse.item())
#     history['loss_reg'].append(reg_log)
#     history['loss_total'].append(total_log)
#     history['loss_teacher'].append(last_teacher_gap)
#     history['keep_ratio'].append(log_ratio)
#     history['lr_backbone'].append(optimizer.param_groups[0]['lr'])
#     history['lr_gate'].append(optimizer.param_groups[1]['lr'])
#     history['temperature'].append(temperature)

#     pbar.set_description(
#         f"{'WU' if in_warmup else '  '} "
#         f"dit={loss_dit.item():.4f} | "
#         f"sp={loss_sparse.item():.4f} | "
#         f"reg={reg_log:.4f} | "
#         f"gap={last_teacher_gap:.4f} | "
#         f"kept={log_ratio*100:.1f}% | "
#         f"bs={BATCH_SIZE} | "
#         f"t={temperature:.2f}"
#     )

#     # ── checkpoint every CHECKPOINT_INTERVAL steps ─────────────────────────────
#     if step % CHECKPOINT_INTERVAL == 0 and step > resume_step:
#         ckpt = {
#             'step'             : step,
#             'model_state'      : dit.state_dict(),
#             'optimizer_state'  : optimizer.state_dict(),
#             'scheduler_state'  : scheduler.state_dict(),
#             'history'          : history,
#             'last_loss_reg'    : last_loss_reg.item(),
#             'last_teacher_gap' : last_teacher_gap,
#             # config
#             'pruning_layers'   : PRUNING_LAYERS,
#             'hidden_dim'       : hidden_dim,
#             'steps_total'      : STEPS,
#             'batch_size'       : BATCH_SIZE,
#             'target_ratio'     : TARGET_RATIO,
#         }
#         torch.save(ckpt, CHECKPOINT_PATH)
#         pbar.write(f"  Checkpoint saved at step {step} → {CHECKPOINT_PATH}")

# print("\nTraining complete.")
# print("=" * 70)
# print(f"  Final loss_dit    : {history['loss_dit'][-1]:.4f}")
# print(f"  Final loss_sparse : {history['loss_sparse'][-1]:.4f}")
# print(f"  Final loss_reg    : {history['loss_reg'][-1]:.4f}")
# print(f"  Final keep_ratio  : {history['keep_ratio'][-1]*100:.1f}%")
# print(f"  Final teacher_gap : {history['loss_teacher'][-1]:.4f}")
# print(f"  Steps trained     : {step + 1}")
# print(f"  History length    : {len(history['loss_dit'])} entries")

# # ── save final checkpoint ──────────────────────────────────────────────────────
# final_ckpt = {
#     'step'             : step,
#     'model_state'      : dit.state_dict(),
#     'optimizer_state'  : optimizer.state_dict(),
#     'scheduler_state'  : scheduler.state_dict(),
#     'history'          : history,
#     'last_loss_reg'    : last_loss_reg.item(),
#     'last_teacher_gap' : last_teacher_gap,
#     'pruning_layers'   : PRUNING_LAYERS,
#     'hidden_dim'       : hidden_dim,
#     'steps_total'      : STEPS,
#     'batch_size'       : BATCH_SIZE,
#     'target_ratio'     : TARGET_RATIO,
# }
# FINAL_PATH = '/workspace/checkpoints/dit_sdt_unfrozen_final.pt'
# torch.save(final_ckpt, FINAL_PATH)
# print(f"  Final checkpoint  : {FINAL_PATH}")

## Cell 9: Training Curves

Visualises all metrics logged during training.
Run immediately after Cell 8 completes.

### What each plot shows

**Loss curves (top row)**
- `loss_dit` — quality signal. Should decrease and stabilise. If it keeps rising the gate is hurting predictions.
- `loss_sparse` — budget signal. Should drop to near zero quickly as the gate finds the 50% target.
- `loss_reg` — regularisation signal. Should stay small. If it rises sharply the backbone is drifting from pretrained weights.
- `loss_total` — weighted sum of all three. Overall training health.

**Gap and ratio (bottom left)**
- `teacher_gap` — MSE between student and teacher noise predictions. Should decrease over training. Measures how much the gate hurts quality relative to the full model.

**Keep ratio (bottom middle)**
- Average fraction of tokens kept across all 14 gated layers. Should converge to 0.5 (TARGET_RATIO). If it stays at 1.0 the sparsity loss is too weak. If it collapses to 0.0 the gate has failed.

**Temperature (bottom right)**
- Gumbel-Softmax temperature annealing from 2.0 to 0.5. Should be a smooth exponential decay. Confirms annealing is working correctly.

### What healthy curves look like on a full 50K run
- `loss_dit` converges within 5K steps
- `loss_sparse` drops to < 0.001 within 1K steps
- `loss_reg` stays below 0.01 throughout
- `teacher_gap` decreases steadily — never rises
- `keep_ratio` stabilises at 0.48–0.52
- Temperature decays smoothly from 2.0 to 0.5

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def plot_training_curves(history, title_suffix=""):
    """
    Plot all training metrics from the history dict.
    Works for both dry run (10 steps) and full run (50K steps).
    """
    steps = list(range(1, len(history['loss_dit']) + 1))
    n     = len(steps)

    # smooth helper — moving average for readability on long runs
    def smooth(values, window=1):
        if n < 50 or window == 1:
            return values
        kernel = np.ones(window) / window
        padded = np.pad(values, (window//2, window//2), mode='edge')
        return np.convolve(padded, kernel, mode='valid')[:n]

    window = max(1, n // 50)   # auto window size — 2% of total steps

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(
        f"Training curves — SDT unfrozen baseline{title_suffix}",
        fontsize=14, fontweight='normal', y=1.01
    )

    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

    # ── Row 0: loss curves ─────────────────────────────────────────────────────
    loss_keys = ['loss_dit', 'loss_sparse', 'loss_reg', 'loss_total']
    loss_colors = ['#378ADD', '#EF9F27', '#1D9E75', '#D85A30']
    loss_labels = ['L_DiT (quality)', 'L_sparse (budget)',
                   'L_reg (regularisation)', 'L_total (weighted sum)']

    for i, (key, color, label) in enumerate(
            zip(loss_keys, loss_colors, loss_labels)):
        ax = fig.add_subplot(gs[0, i])
        raw    = history[key]
        smoothed = smooth(raw, window)
        if window > 1:
            ax.plot(steps, raw, alpha=0.25, color=color, linewidth=0.8)
        ax.plot(steps, smoothed, color=color, linewidth=1.8, label=label)
        ax.set_title(label, fontsize=11)
        ax.set_xlabel("Step", fontsize=9)
        ax.set_ylabel("Loss", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        # annotate final value
        ax.annotate(
            f"final: {raw[-1]:.4f}",
            xy=(steps[-1], raw[-1]),
            xytext=(-40, 10),
            textcoords='offset points',
            fontsize=8,
            color=color
        )

    # ── Row 1: gap, keep ratio, temperature ───────────────────────────────────
    # teacher gap
    ax_gap = fig.add_subplot(gs[1, 0:2])
    raw_gap = history['loss_teacher']
    ax_gap.plot(steps, smooth(raw_gap, window),
                color='#534AB7', linewidth=1.8)
    if window > 1:
        ax_gap.plot(steps, raw_gap, alpha=0.2,
                    color='#534AB7', linewidth=0.8)
    ax_gap.axhline(0, color='gray', linestyle='--',
                   linewidth=0.8, alpha=0.5)
    ax_gap.set_title("Teacher gap  (student vs teacher MSE)", fontsize=11)
    ax_gap.set_xlabel("Step", fontsize=9)
    ax_gap.set_ylabel("MSE", fontsize=9)
    ax_gap.tick_params(labelsize=8)
    ax_gap.grid(True, alpha=0.3, linewidth=0.5)
    ax_gap.annotate(
        f"final: {raw_gap[-1]:.4f}",
        xy=(steps[-1], raw_gap[-1]),
        xytext=(-50, 10),
        textcoords='offset points',
        fontsize=8, color='#534AB7'
    )

    # keep ratio
    ax_kr = fig.add_subplot(gs[1, 2])
    raw_kr = history['keep_ratio']
    ax_kr.plot(steps, smooth(raw_kr, window),
               color='#1D9E75', linewidth=1.8)
    if window > 1:
        ax_kr.plot(steps, raw_kr, alpha=0.2,
                   color='#1D9E75', linewidth=0.8)
    ax_kr.axhline(0.5, color='#EF9F27', linestyle='--',
                  linewidth=1.2, label='Target 50%')
    ax_kr.axhline(0.45, color='gray', linestyle=':',
                  linewidth=0.8, alpha=0.6)
    ax_kr.axhline(0.55, color='gray', linestyle=':',
                  linewidth=0.8, alpha=0.6)
    ax_kr.set_ylim(0, 1)
    ax_kr.set_title("Token keep ratio", fontsize=11)
    ax_kr.set_xlabel("Step", fontsize=9)
    ax_kr.set_ylabel("Fraction kept", fontsize=9)
    ax_kr.tick_params(labelsize=8)
    ax_kr.grid(True, alpha=0.3, linewidth=0.5)
    ax_kr.legend(fontsize=8)
    ax_kr.annotate(
        f"final: {raw_kr[-1]*100:.1f}%",
        xy=(steps[-1], raw_kr[-1]),
        xytext=(-55, 10),
        textcoords='offset points',
        fontsize=8, color='#1D9E75'
    )

    # temperature
    ax_t = fig.add_subplot(gs[1, 3])
    ax_t.plot(steps, history['temperature'],
              color='#D85A30', linewidth=1.8)
    ax_t.axhline(TEMP_END, color='gray', linestyle='--',
                 linewidth=0.8, alpha=0.6, label=f'Target {TEMP_END}')
    ax_t.set_ylim(0, TEMP_START + 0.2)
    ax_t.set_title("Gumbel temperature", fontsize=11)
    ax_t.set_xlabel("Step", fontsize=9)
    ax_t.set_ylabel("Temperature τ", fontsize=9)
    ax_t.tick_params(labelsize=8)
    ax_t.grid(True, alpha=0.3, linewidth=0.5)
    ax_t.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: training_curves.png")


# ── Summary table ──────────────────────────────────────────────────────────────
def print_training_summary(history):
    n = len(history['loss_dit'])
    print("=" * 52)
    print(f"  Training summary ({n} steps)")
    print("=" * 52)
    print(f"  loss_dit    final : {history['loss_dit'][-1]:.4f}")
    print(f"  loss_sparse final : {history['loss_sparse'][-1]:.4f}")
    print(f"  loss_reg    final : {history['loss_reg'][-1]:.4f}")
    print(f"  loss_total  final : {history['loss_total'][-1]:.4f}")
    print(f"  teacher_gap final : {history['loss_teacher'][-1]:.4f}")
    print(f"  keep_ratio  final : {history['keep_ratio'][-1]*100:.1f}%")
    print(f"  temperature final : {history['temperature'][-1]:.2f}")
    print("-" * 52)
    if n >= 10:
        print(f"  loss_dit    min   : {min(history['loss_dit']):.4f}")
        print(f"  teacher_gap min   : {min(history['loss_teacher']):.4f}")
        print(f"  keep_ratio  mean  : {np.mean(history['keep_ratio'])*100:.1f}%")
    print("=" * 52)


# ── Run ────────────────────────────────────────────────────────────────────────
suffix = " [dry run — 10 steps]" if DRY_RUN else f" [{STEPS:,} steps]"
plot_training_curves(history, title_suffix=suffix)
print_training_summary(history)

## Cell 10: FLOP Benchmark

Measures computational cost of teacher (full DiT) vs student (pruned DiT).
Reports three numbers:

- **Teacher GMACs** — baseline full model compute
- **Student GMACs** — pruned model compute (fvcore measurement)
- **Wall-clock speedup** — actual GPU timing over 50 forward passes

### Why GMACs and speedup differ
fvcore counts multiply-accumulate operations theoretically.
It cannot fully trace dynamic control flow (the per-image gather/scatter loop)
so it underestimates student savings. Wall-clock timing measures real GPU
execution and is the more honest number for a thesis.

### Expected numbers at 14 gated layers
- FLOP reduction: ~7-10% (fvcore undercount)
- Wall-clock speedup: 1.05-1.15× at batch size 1
- Speedup grows with batch size and number of gated layers

### RunPod note
Run at batch size 32 for a more representative speedup measurement.
Single-sample speedup underestimates real throughput gains.

In [ ]:
from fvcore.nn import FlopCountAnalysis
import time

# ── Benchmark config ───────────────────────────────────────────────────────────
BENCH_BATCH = 1      # RUNPOD: increase to 32 for representative measurement
BENCH_REPS  = 50     # number of forward passes for timing average
WARMUP_REPS = 10     # GPU warmup passes before timing

# ── Benchmark inputs ───────────────────────────────────────────────────────────
z_bench = torch.randn(BENCH_BATCH, 4, 32, 32, device=device)
t_bench = torch.randint(0, 1000, (BENCH_BATCH,), device=device)
y_bench = torch.zeros(BENCH_BATCH, dtype=torch.long, device=device)

# ── fvcore FLOP count ──────────────────────────────────────────────────────────
print("Computing FLOPs via fvcore...")

# teacher FLOPs — pruning off
set_pruning_mode(dit, enabled=False)
dit.eval()

try:
    flops_teacher = FlopCountAnalysis(dit, (z_bench, t_bench, y_bench))
    flops_teacher.unsupported_ops_warnings(False)
    flops_teacher.uncalled_modules_warnings(False)
    gmac_teacher  = flops_teacher.total() / 1e9
except Exception as e:
    print(f"  fvcore teacher error: {e}")
    gmac_teacher = None

# student FLOPs — pruning on
set_pruning_mode(dit, enabled=True)
dit.eval()
for pp in all_parasites:
    pp.eval()

try:
    flops_student = FlopCountAnalysis(dit, (z_bench, t_bench, y_bench))
    flops_student.unsupported_ops_warnings(False)
    flops_student.uncalled_modules_warnings(False)
    gmac_student  = flops_student.total() / 1e9
except Exception as e:
    print(f"  fvcore student error: {e}")
    gmac_student = None

# ── Wall-clock timing ──────────────────────────────────────────────────────────
print("Measuring wall-clock latency...")

starter = torch.cuda.Event(enable_timing=True)
ender   = torch.cuda.Event(enable_timing=True)

# teacher timing
set_pruning_mode(dit, enabled=False)
dit.eval()
with torch.no_grad():
    for _ in range(WARMUP_REPS):
        _ = dit(z_bench, t_bench, y_bench)
    torch.cuda.synchronize()
    starter.record()
    for _ in range(BENCH_REPS):
        _ = dit(z_bench, t_bench, y_bench)
    ender.record()
torch.cuda.synchronize()
ms_teacher = starter.elapsed_time(ender) / BENCH_REPS

# student timing
set_pruning_mode(dit, enabled=True)
dit.eval()
with torch.no_grad():
    for _ in range(WARMUP_REPS):
        _ = dit(z_bench, t_bench, y_bench)
    torch.cuda.synchronize()
    starter.record()
    for _ in range(BENCH_REPS):
        _ = dit(z_bench, t_bench, y_bench)
    ender.record()
torch.cuda.synchronize()
ms_student = starter.elapsed_time(ender) / BENCH_REPS

# ── Throughput ─────────────────────────────────────────────────────────────────
imgs_per_sec_teacher = (BENCH_BATCH * 1000) / ms_teacher
imgs_per_sec_student = (BENCH_BATCH * 1000) / ms_student

# ── Results ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 52)
print("  FLOP Benchmark Results")
print("=" * 52)

if gmac_teacher and gmac_student:
    reduction = (1 - gmac_student / gmac_teacher) * 100
    print(f"  Teacher GMACs     : {gmac_teacher:.2f}")
    print(f"  Student GMACs     : {gmac_student:.2f}")
    print(f"  FLOP reduction    : {reduction:.1f}%")
else:
    print("  GMACs             : fvcore trace failed")

print(f"  Teacher latency   : {ms_teacher:.2f} ms/batch")
print(f"  Student latency   : {ms_student:.2f} ms/batch")
print(f"  Wall-clock speedup: {ms_teacher/ms_student:.3f}×")
print(f"  Teacher throughput: {imgs_per_sec_teacher:.1f} img/s")
print(f"  Student throughput: {imgs_per_sec_student:.1f} img/s")
print(f"  Batch size        : {BENCH_BATCH}")
print(f"  Reps averaged     : {BENCH_REPS}")
print("-" * 52)
print(f"  Gated layers      : {len(PRUNING_LAYERS)} / 28")
print(f"  Target keep ratio : {TARGET_RATIO*100:.0f}%")
print(f"  Actual keep ratio : {history['keep_ratio'][-1]*100:.1f}%")
print("=" * 52)

# ── Context for thesis ─────────────────────────────────────────────────────────
print("""
  Note for thesis:
  fvcore underestimates student savings — it cannot trace the
  dynamic gather/scatter branches in PruningWrapper.forward().
  Wall-clock speedup is the more reliable efficiency metric.
  Extend to all 28 layers for ~2× the current FLOP reduction.
""")

# restore train mode
dit.train()
for pp in all_parasites:
    pp.train()

## Cell 11: Qualitative Evaluation

Generates images with both teacher and student and compares them visually
and quantitatively.

### Metrics reported

| Metric | Measures | Good value |
|---|---|---|
| PSNR | Pixel-level fidelity (dB) | > 30 dB |
| SSIM | Structural similarity | > 0.90 |
| MSE | Raw pixel difference | < 0.05 |

### What to expect

**Dry run (10 steps):** metrics will be poor — PSNR ~12dB, SSIM ~0.28.
The gate has barely trained. This is expected — the cell exists to verify
the pipeline runs end to end, not to show good results.

**Full run (50K steps, unfrozen):** metrics should improve significantly
vs the frozen baseline because the backbone co-adapts with the gate.
Target: PSNR > 20dB, SSIM > 0.70 at 50% token keep ratio.

### Per-layer mask visualisation
Bottom row shows the binary token mask for each of the 14 gated layers.
Yellow = token kept (MLP runs), black = token skipped (MLP bypassed).
On a well-trained gate bright regions should roughly track the foreground object.

In [ ]:
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from diffusion import create_diffusion

GRID = 16   # 16×16 = 256 tokens for DiT-XL/2

def to_img(tensor):
    """Convert VAE output tensor to HxWxC numpy array in [0,1]."""
    img = tensor[0].permute(1, 2, 0).cpu().float().numpy()
    return np.clip((img + 1) / 2, 0, 1)


def compute_metrics(img_t, img_s):
    """
    img_t, img_s: HxWxC float32 numpy arrays in [0,1]
    returns: psnr (dB), ssim, mse
    """
    p   = psnr_fn(img_t, img_s, data_range=1.0)
    s   = ssim_fn(img_t, img_s, data_range=1.0, channel_axis=2)
    mse = float(np.mean((img_t - img_s) ** 2))
    return p, s, mse


def run_inference(class_label=208, seed=42, ddim_steps=50):
    """
    Run full DDIM denoising with teacher and student.
    Returns decoded images, gate masks, and quality metrics.
    """
    torch.manual_seed(seed)
    z    = torch.randn(1, 4, 32, 32, device=device)
    y    = torch.tensor([class_label], device=device)
    diff = create_diffusion(timestep_respacing=f"ddim{ddim_steps}")

    dit.eval()
    for pp in all_parasites:
        pp.eval()

    # teacher
    set_pruning_mode(dit, enabled=False)
    with torch.no_grad():
        lat_t = diff.ddim_sample_loop(
            dit, shape=(1, 4, 32, 32),
            noise=z.clone(),
            model_kwargs={"y": y},
            device=device, progress=False
        )
        px_t = vae.decode(lat_t / 0.18215).sample

    # student
    set_pruning_mode(dit, enabled=True)
    with torch.no_grad():
        lat_s = diff.ddim_sample_loop(
            dit, shape=(1, 4, 32, 32),
            noise=z.clone(),
            model_kwargs={"y": y},
            device=device, progress=False
        )
        px_s = vae.decode(lat_s / 0.18215).sample

    masks = get_all_masks(dit)
    kept  = sum(m.mean().item() for m in masks) / len(masks) * 100

    img_t_np = to_img(px_t)
    img_s_np = to_img(px_s)
    p, s, mse = compute_metrics(img_t_np, img_s_np)

    return px_t, px_s, masks, kept, p, s, mse


def plot_qualitative(px_t, px_s, masks, kept, psnr_val, ssim_val, mse_val,
                     class_label, seed):
    n_layers = len(masks)
    n_cols   = max(n_layers, 3)

    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(max(4 * n_cols, 16), 9)
    )

    # ── row 0: teacher | student | metric card | empty ─────────────────────────
    axes[0, 0].imshow(to_img(px_t))
    axes[0, 0].set_title(
        f"Teacher (full)\nClass {class_label}", fontsize=10
    )
    axes[0, 0].axis('off')

    axes[0, 1].imshow(to_img(px_s))
    axes[0, 1].set_title(
        f"Student (pruned)\n{kept:.1f}% tokens kept", fontsize=10
    )
    axes[0, 1].axis('off')

    # metric card
    axes[0, 2].axis('off')
    badge_color = '#2ecc71' if psnr_val > 25 else \
                  '#f39c12' if psnr_val > 15 else '#e74c3c'
    quality     = 'Good' if psnr_val > 25 else \
                  'Fair' if psnr_val > 15 else 'Poor (expected at low steps)'
    metrics_text = (
        f"Pixel MSE\n{mse_val:.5f}\n\n"
        f"PSNR\n{psnr_val:.2f} dB\n\n"
        f"SSIM\n{ssim_val:.4f}\n\n"
        f"Tokens kept\n{kept:.1f}%\n\n"
        f"Quality\n{quality}"
    )
    axes[0, 2].text(
        0.5, 0.5, metrics_text,
        transform=axes[0, 2].transAxes,
        fontsize=11, va='center', ha='center',
        bbox=dict(
            boxstyle='round,pad=0.6',
            facecolor='#f8f9fa',
            edgecolor=badge_color,
            linewidth=2
        ),
        linespacing=1.8
    )
    axes[0, 2].set_title("Quality metrics\n(student vs teacher)", fontsize=10)

    for ax in axes[0, 3:]:
        ax.axis('off')

    # ── row 1: per-layer gate masks ────────────────────────────────────────────
    for li, mask in enumerate(masks):
        grid = mask[0].cpu().float().numpy().reshape(GRID, GRID)
        im   = axes[1, li].imshow(
            grid, cmap='magma',
            interpolation='nearest', vmin=0, vmax=1
        )
        kept_pct = mask[0].float().mean().item() * 100
        axes[1, li].set_title(
            f"L{PRUNING_LAYERS[li]}\n{kept_pct:.0f}% kept",
            fontsize=9
        )
        axes[1, li].axis('off')
        plt.colorbar(im, ax=axes[1, li], shrink=0.7)

    for ax in axes[1, n_layers:]:
        ax.axis('off')

    plt.suptitle(
        f"Seed {seed} | Class {class_label} | "
        f"PSNR {psnr_val:.2f}dB | SSIM {ssim_val:.4f} | "
        f"MSE {mse_val:.5f} | {'Unfrozen' if True else 'Frozen'} backbone",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig("qualitative_eval.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: qualitative_eval.png")


# ── Run evaluation ─────────────────────────────────────────────────────────────
CLASS_LABEL  = 208
SEED         = 42
DDIM_STEPS   = 50     # RUNPOD: keep at 50 — matches thesis evaluation protocol

print(f"Running inference — class {CLASS_LABEL}, seed {SEED}, "
      f"{DDIM_STEPS} DDIM steps...")

px_t, px_s, masks, kept, psnr_val, ssim_val, mse_val = run_inference(
    class_label=CLASS_LABEL,
    seed=SEED,
    ddim_steps=DDIM_STEPS
)

print("=" * 52)
print(f"  PSNR        : {psnr_val:.2f} dB")
print(f"  SSIM        : {ssim_val:.4f}")
print(f"  Pixel MSE   : {mse_val:.5f}")
print(f"  Tokens kept : {kept:.1f}%")
print("=" * 52)

plot_qualitative(
    px_t, px_s, masks, kept,
    psnr_val, ssim_val, mse_val,
    CLASS_LABEL, SEED
)

# restore train mode for subsequent cells
dit.train()
for pp in all_parasites:
    pp.train()

## Cell 12: Per-Layer Analysis

Analyses how the gate behaves across the 14 gated layers individually.
Two visualisations:

**Keep ratio bar chart**
Shows what fraction of tokens each layer keeps.
A well-trained gate should show variation — some layers prune aggressively,
others conservatively, depending on how much redundancy exists at that depth.
Uniform 50% across all layers means the gate hasn't learned layer-specific behaviour.

**Token mask grid**
Shows the spatial pattern of kept tokens at each layer for one image.
On a well-trained gate:
- Early layers (0, 2): less structured — fewer contextual features available
- Middle layers (8-16): spatially meaningful — should roughly track foreground
- Late layers (22-26): concentrated — only critical tokens processed

### What this proves for the thesis
Different layers have different optimal keep ratios.
A uniform 50% budget across all layers is suboptimal.
MAPPO can learn per-layer budgets naturally through the reward signal —
this chart motivates that design choice.

In [ ]:
def plot_per_layer_analysis(masks, pruning_layers, grid=16):
    """
    Two-panel per-layer analysis.
    masks         : list of [1, N] tensors from get_all_masks()
    pruning_layers: list of layer indices
    grid          : token grid size (16 for DiT-XL/2)
    """
    n_layers = len(masks)

    # ── compute per-layer statistics ───────────────────────────────────────────
    keep_ratios = [m[0].float().mean().item() * 100 for m in masks]
    mean_ratio  = np.mean(keep_ratios)
    std_ratio   = np.std(keep_ratios)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        "Per-layer gate analysis — SDT unfrozen baseline",
        fontsize=13, y=1.01
    )

    gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.5)

    # ── panel 1: keep ratio bar chart ──────────────────────────────────────────
    ax_bar = fig.add_subplot(gs[0])

    colors = []
    for kr in keep_ratios:
        if kr < 35:
            colors.append('#E24B4A')   # aggressive pruning — red
        elif kr > 65:
            colors.append('#378ADD')   # conservative — blue
        else:
            colors.append('#1D9E75')   # on target — green

    bars = ax_bar.bar(
        range(n_layers), keep_ratios,
        color=colors, alpha=0.85,
        edgecolor='white', linewidth=0.5
    )

    # target and mean lines
    ax_bar.axhline(
        50, color='#EF9F27', linestyle='--',
        linewidth=1.5, label='Target 50%', zorder=3
    )
    ax_bar.axhline(
        mean_ratio, color='#534AB7', linestyle=':',
        linewidth=1.5, label=f'Mean {mean_ratio:.1f}%', zorder=3
    )
    ax_bar.fill_between(
        [-0.5, n_layers - 0.5],
        mean_ratio - std_ratio,
        mean_ratio + std_ratio,
        alpha=0.1, color='#534AB7', label=f'±1σ ({std_ratio:.1f}%)'
    )

    # value labels on bars
    for i, (bar, kr) in enumerate(zip(bars, keep_ratios)):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f'{kr:.0f}%',
            ha='center', va='bottom',
            fontsize=8, color='#333333'
        )

    ax_bar.set_xticks(range(n_layers))
    ax_bar.set_xticklabels(
        [f'L{i}' for i in pruning_layers],
        fontsize=9
    )
    ax_bar.set_ylim(0, 110)
    ax_bar.set_ylabel("Tokens kept (%)", fontsize=10)
    ax_bar.set_xlabel("Gated layer", fontsize=10)
    ax_bar.set_title(
        "Per-layer token keep ratio  "
        "(green=on target | red=over-pruning | blue=under-pruning)",
        fontsize=10
    )
    ax_bar.legend(fontsize=9, loc='upper right')
    ax_bar.grid(True, axis='y', alpha=0.3, linewidth=0.5)

    # ── panel 2: spatial mask grid ─────────────────────────────────────────────
    n_cols = n_layers
    gs2    = gridspec.GridSpecFromSubplotSpec(
        1, n_cols, subplot_spec=gs[1], wspace=0.05
    )

    for li, (mask, layer_idx) in enumerate(zip(masks, pruning_layers)):
        ax = fig.add_subplot(gs2[li])
        grid_map = mask[0].cpu().float().numpy().reshape(grid, grid)

        ax.imshow(
            grid_map, cmap='magma',
            interpolation='nearest', vmin=0, vmax=1
        )
        ax.set_title(
            f"L{layer_idx}\n{keep_ratios[li]:.0f}%",
            fontsize=8,
            color='#E24B4A' if keep_ratios[li] < 35
            else '#378ADD' if keep_ratios[li] > 65
            else '#1D9E75'
        )
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("per_layer_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: per_layer_analysis.png")


def print_per_layer_table(masks, pruning_layers):
    """Print a clean per-layer statistics table."""
    keep_ratios = [m[0].float().mean().item() * 100 for m in masks]

    print("=" * 42)
    print(f"  {'Layer':>6} | {'Keep %':>7} | {'Status':>18}")
    print("-" * 42)
    for li, (layer_idx, kr) in enumerate(zip(pruning_layers, keep_ratios)):
        if kr < 35:
            status = "over-pruning  ⚠"
        elif kr > 65:
            status = "under-pruning ⚠"
        else:
            status = "on target     ✓"
        print(f"  {layer_idx:>6} | {kr:>6.1f}% | {status:>18}")
    print("-" * 42)
    print(f"  {'Mean':>6} | {np.mean(keep_ratios):>6.1f}% |")
    print(f"  {'Std':>6} | {np.std(keep_ratios):>6.1f}% |")
    print(f"  {'Min':>6} | {min(keep_ratios):>6.1f}% |")
    print(f"  {'Max':>6} | {max(keep_ratios):>6.1f}% |")
    print("=" * 42)


# ── collect masks via one student forward pass ─────────────────────────────────
print("Collecting per-layer gate masks...")

z_eval  = latent_cache[0:1].to(device)
t_eval  = torch.randint(0, 1000, (1,), device=device)
y_eval  = label_cache[0:1].to(device)

set_pruning_mode(dit, enabled=True)
dit.eval()
for pp in all_parasites:
    pp.eval()

with torch.no_grad():
    _ = dit(z_eval, t_eval, y_eval)

masks = get_all_masks(dit, live=False)

print(f"  Collected {len(masks)} layer masks")
print(f"  Mask shape: {masks[0].shape}")

# ── plot and table ─────────────────────────────────────────────────────────────
plot_per_layer_analysis(masks, PRUNING_LAYERS)
print_per_layer_table(masks, PRUNING_LAYERS)

# restore train mode
dit.train()
for pp in all_parasites:
    pp.train()

## Cell 13: Saliency Experiments

Runs all four saliency experiments using the persistent HookManager.
No hooks are registered or removed here — they were registered once in Cell 5.
Each experiment activates the relevant group, runs a forward pass, reads
the data, then deactivates.

### Four experiments

| # | Name | What it measures | Key finding |
|---|---|---|---|
| 1 | Cross-timestep | How importance changes across t=900→100 | Diffuse→concentrated transition |
| 2 | Gate vs saliency (IoU) | How well gate decisions match ground truth | Baseline IoU to beat with MAPPO |
| 3 | Per-layer entropy | Saliency pattern at each of 28 layers | Which layers are prunable |
| 4 | DDPM vs DDIM | Schedule mismatch between training and inference | Motivates MAPPO under DDIM |

### Unfrozen vs frozen comparison
Run these experiments after both frozen (SDT_only.ipynb) and unfrozen training.
If IoU improves with unfrozen backbone, co-adaptation is working.
If IoU is similar, the improvement is purely from MAPPO (Phase 2).
Both outcomes are valid thesis findings.

### Hook usage pattern
```python

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GRID       = 16
N_TOKENS   = GRID * GRID
SALIENCY_CLASS  = 208
SALIENCY_SEED   = 42
SALIENCY_IMG    = 0

torch.manual_seed(SALIENCY_SEED)
x0_sal = latent_cache[SALIENCY_IMG:SALIENCY_IMG+1].to(device)
y_sal  = torch.tensor([SALIENCY_CLASS], device=device)


# ── Saliency helpers ───────────────────────────────────────────────────────────
def get_noisy_latent(x0, t_val):
    t_tensor = torch.tensor([t_val], device=device)
    noise    = torch.randn_like(x0)
    z_t      = diffusion.q_sample(x0, t_tensor, noise=noise)
    return z_t, noise


def compute_gradient_saliency(z_t, t_tensor):
    """Gradient saliency via HookManager gradient group."""
    dit.eval()
    set_pruning_mode(dit, enabled=False)

    hook_manager.activate('gradient')
    hook_manager.clear('gradient')

    with torch.enable_grad():
        out        = dit(z_t, t_tensor, y_sal)
        noise_pred = out[:, :IN_CHANNELS, :, :]
        loss       = F.mse_loss(noise_pred, torch.zeros_like(noise_pred))
        loss.backward()

    storage = hook_manager.get('gradient')
    hook_manager.deactivate('gradient')

    if not storage or storage[0].grad is None:
        return np.zeros((GRID, GRID))

    grad = storage[0].grad[0]
    sal  = grad.norm(dim=-1).detach().cpu().numpy()
    sal  = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    return sal.reshape(GRID, GRID)


def compute_attention_entropy(z_t, t_tensor):
    """Attention entropy saliency via HookManager attention group."""
    dit.eval()
    set_pruning_mode(dit, enabled=False)

    hook_manager.activate('attention')
    hook_manager.clear('attention')

    with torch.no_grad():
        _ = dit(z_t, t_tensor, y_sal)

    storage = hook_manager.get('attention')
    hook_manager.deactivate('attention')

    if not storage:
        return np.zeros((GRID, GRID))

    avg_entropy = torch.stack(storage).mean(dim=0).numpy()
    importance  = avg_entropy.max() - avg_entropy
    importance  = (importance - importance.min()) / \
                  (importance.max() - importance.min() + 1e-8)
    return importance.reshape(GRID, GRID)


def get_gate_mask(layer_idx):
    block = dit.blocks[layer_idx]
    if isinstance(block, PruningWrapper) and block._last_mask is not None:
        return block._last_mask[0].cpu().float().numpy().reshape(GRID, GRID)
    return np.ones((GRID, GRID))


# ══════════════════════════════════════════════════════════════════════════════
# Experiment 1 — Cross-timestep saliency
# ══════════════════════════════════════════════════════════════════════════════
print("Experiment 1: Cross-timestep saliency...")
TIMESTEPS = [900, 700, 500, 300, 100]

grad_maps = []
attn_maps = []

for t_val in TIMESTEPS:
    print(f"  t={t_val}...", end=' ')
    z_t, _   = get_noisy_latent(x0_sal, t_val)
    t_tensor = torch.tensor([t_val], device=device)
    grad_maps.append(compute_gradient_saliency(z_t, t_tensor))
    attn_maps.append(compute_attention_entropy(z_t, t_tensor))
    print("done")

# decode reference image
with torch.no_grad():
    ref_img = vae.decode(x0_sal / 0.18215).sample
ref_np = np.clip(
    (ref_img[0].permute(1,2,0).cpu().float().numpy() + 1) / 2, 0, 1
)

fig, axes = plt.subplots(
    3, len(TIMESTEPS) + 1,
    figsize=(4*(len(TIMESTEPS)+1), 12)
)
axes[0,0].imshow(ref_np)
axes[0,0].set_title(f"Original\nClass {SALIENCY_CLASS}", fontsize=10)
axes[0,0].axis('off')
axes[1,0].axis('off')
axes[1,0].text(0.5, 0.5, "Gradient\nsaliency",
               ha='center', va='center', fontsize=11,
               transform=axes[1,0].transAxes)
axes[2,0].axis('off')
axes[2,0].text(0.5, 0.5, "Attention\nentropy",
               ha='center', va='center', fontsize=11,
               transform=axes[2,0].transAxes)

for col, (t_val, g, a) in enumerate(
        zip(TIMESTEPS, grad_maps, attn_maps), start=1):
    z_t, _ = get_noisy_latent(x0_sal, t_val)
    with torch.no_grad():
        nd = vae.decode(z_t / 0.18215).sample
    noisy_np = np.clip(
        (nd[0].permute(1,2,0).cpu().float().numpy() + 1) / 2, 0, 1
    )
    axes[0,col].imshow(noisy_np)
    axes[0,col].set_title(f"t = {t_val}", fontsize=10)
    axes[0,col].axis('off')
    im1 = axes[1,col].imshow(g, cmap='hot',
                              interpolation='nearest', vmin=0, vmax=1)
    axes[1,col].axis('off')
    plt.colorbar(im1, ax=axes[1,col], shrink=0.8)
    im2 = axes[2,col].imshow(a, cmap='hot',
                              interpolation='nearest', vmin=0, vmax=1)
    axes[2,col].axis('off')
    plt.colorbar(im2, ax=axes[2,col], shrink=0.8)

plt.suptitle(
    f"Experiment 1: Cross-timestep saliency | "
    f"Class {SALIENCY_CLASS} | Seed {SALIENCY_SEED} | Unfrozen backbone",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("exp1_cross_timestep.png", dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: exp1_cross_timestep.png")


# ══════════════════════════════════════════════════════════════════════════════
# Experiment 2 — Gate mask vs saliency (IoU)
# ══════════════════════════════════════════════════════════════════════════════
print("\nExperiment 2: Gate vs saliency IoU...")
THRESHOLD = 0.5
t_val     = 500
z_t, _    = get_noisy_latent(x0_sal, t_val)
t_tensor  = torch.tensor([t_val], device=device)

grad_map = compute_gradient_saliency(z_t, t_tensor)

# populate gate masks with one student forward pass
set_pruning_mode(dit, enabled=True)
dit.eval()
with torch.no_grad():
    _ = dit(z_t.expand(1,-1,-1,-1), t_tensor, y_sal)

iou_scores = []
fig, axes  = plt.subplots(
    3, len(PRUNING_LAYERS),
    figsize=(3*len(PRUNING_LAYERS), 9)
)

for li, layer_idx in enumerate(PRUNING_LAYERS):
    gate_mask  = get_gate_mask(layer_idx)
    sal_binary = (grad_map > THRESHOLD).astype(float)
    intersection = (sal_binary * gate_mask).sum()
    union        = np.clip(sal_binary + gate_mask, 0, 1).sum()
    iou          = intersection / (union + 1e-8)
    iou_scores.append(iou)

    axes[0,li].imshow(grad_map, cmap='hot',
                       interpolation='nearest', vmin=0, vmax=1)
    axes[0,li].set_title(f"L{layer_idx}\nSaliency", fontsize=8)
    axes[0,li].axis('off')

    axes[1,li].imshow(gate_mask, cmap='RdYlGn',
                       interpolation='nearest', vmin=0, vmax=1)
    axes[1,li].set_title(
        f"Gate\n{gate_mask.mean()*100:.0f}% kept", fontsize=8
    )
    axes[1,li].axis('off')

    overlay = np.zeros((*gate_mask.shape, 3))
    overlay[...,1] = sal_binary * gate_mask          # green: correct keep
    overlay[...,0] = sal_binary * (1 - gate_mask)    # red: missed important
    overlay[...,2] = (1-sal_binary)*(1-gate_mask)    # blue: correct skip
    axes[2,li].imshow(overlay, interpolation='nearest')
    axes[2,li].set_title(f"IoU: {iou:.2f}", fontsize=8)
    axes[2,li].axis('off')

mean_iou = np.mean(iou_scores)
plt.suptitle(
    f"Experiment 2: Gate vs Saliency | t={t_val} | "
    f"Mean IoU: {mean_iou:.3f} | Unfrozen backbone",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig("exp2_gate_vs_saliency.png", dpi=150, bbox_inches='tight')
plt.show()

# IoU bar chart
fig2, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    [f"L{i}" for i in PRUNING_LAYERS], iou_scores,
    color=['#2ecc71' if s > 0.5 else
           '#f39c12' if s > 0.33 else '#e74c3c'
           for s in iou_scores]
)
ax.axhline(mean_iou, color='black', linestyle='--',
           linewidth=1.5, label=f'Mean IoU = {mean_iou:.3f}')
ax.axhline(0.33, color='gray', linestyle=':',
           linewidth=1.2, label='Random chance ≈ 0.33')
ax.axhline(0.5, color='#2ecc71', linestyle=':',
           linewidth=1.2, alpha=0.7, label='Target IoU = 0.50')
ax.set_ylim(0, 1)
ax.set_xlabel("Gated layer")
ax.set_ylabel("IoU")
ax.set_title(
    "Per-layer gate alignment with gradient saliency  "
    "(green > 0.5 | orange > 0.33 | red < 0.33)"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("exp2_iou_barchart.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"  Mean IoU : {mean_iou:.4f}  (random chance ≈ 0.33)")
print(f"  Per-layer: {[f'L{i}:{s:.2f}' for i,s in zip(PRUNING_LAYERS,iou_scores)]}")
print("  Saved: exp2_gate_vs_saliency.png, exp2_iou_barchart.png")


# ══════════════════════════════════════════════════════════════════════════════
# Experiment 3 — Per-layer attention entropy
# ══════════════════════════════════════════════════════════════════════════════
print("\nExperiment 3: Per-layer attention entropy...")
t_val    = 500
z_t, _   = get_noisy_latent(x0_sal, t_val)
t_tensor = torch.tensor([t_val], device=device)

per_layer_sal = []

for target_layer in range(len(dit.blocks)):
    storage  = []
    block    = dit.blocks[target_layer]
    b        = block.block if isinstance(block, PruningWrapper) else block

    def make_single_hook(s):
        def hook(module, inp, out):
            x       = inp[0]
            B, N, D = x.shape
            qkv     = module.qkv(x)
            qkv     = qkv.reshape(B, N, 3, module.num_heads,
                                  D//module.num_heads).permute(2,0,3,1,4)
            q, k, _ = qkv.unbind(0)
            scale   = q.shape[-1] ** -0.5
            attn    = (q @ k.transpose(-2,-1)) * scale
            attn    = attn.softmax(dim=-1)
            eps     = 1e-8
            ent     = -(attn*(attn+eps).log()).sum(dim=-1).mean(dim=1)
            s.append(ent[0].detach().cpu())
        return hook

    handle = b.attn.register_forward_hook(make_single_hook(storage))
    set_pruning_mode(dit, enabled=False)
    dit.eval()
    with torch.no_grad():
        _ = dit(z_t, t_tensor, y_sal)
    handle.remove()

    if storage:
        ent = storage[0].numpy()
        imp = ent.max() - ent
        imp = (imp - imp.min()) / (imp.max() - imp.min() + 1e-8)
        per_layer_sal.append(imp.reshape(GRID, GRID))
    else:
        per_layer_sal.append(np.zeros((GRID, GRID)))

n_layers = len(per_layer_sal)
n_cols   = 7
n_rows   = (n_layers + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(3*n_cols, 3*n_rows))
axes = axes.flatten()

for li, sal in enumerate(per_layer_sal):
    im    = axes[li].imshow(sal, cmap='hot',
                             interpolation='nearest', vmin=0, vmax=1)
    label = f"L{li}" + (" *" if li in PRUNING_LAYERS else "")
    color = '#2ecc71' if li in PRUNING_LAYERS else '#333333'
    axes[li].set_title(label, fontsize=9, color=color)
    axes[li].axis('off')

for ax in axes[n_layers:]:
    ax.axis('off')

plt.suptitle(
    f"Experiment 3: Per-layer attention entropy | t={t_val} | "
    f"* = gated layer | Unfrozen backbone",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("exp3_per_layer.png", dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: exp3_per_layer.png")


# ══════════════════════════════════════════════════════════════════════════════
# Experiment 4 — DDPM vs DDIM schedule comparison
# ══════════════════════════════════════════════════════════════════════════════
print("\nExperiment 4: DDPM vs DDIM schedule comparison...")
from diffusion import create_diffusion as create_diff

DDPM_STEPS   = [100, 300, 500, 700, 900]
ddim_t_vals  = [900, 700, 500, 300, 100]
ddpm_saliency = []
ddim_saliency = []

print("  DDPM saliency...")
for t_val in DDPM_STEPS:
    z_t, _ = get_noisy_latent(x0_sal, t_val)
    t_tensor = torch.tensor([t_val], device=device)
    sal = compute_attention_entropy(z_t, t_tensor)
    ddpm_saliency.append((t_val, sal))
    print(f"    t={t_val} done")

print("  DDIM saliency...")
for t_val in ddim_t_vals:
    z_t, _ = get_noisy_latent(x0_sal, t_val)
    t_tensor = torch.tensor([t_val], device=device)
    sal = compute_attention_entropy(z_t, t_tensor)
    ddim_saliency.append((t_val, sal))
    print(f"    t={t_val} done")

correlations = []
for (t_d, sal_d), (t_i, sal_i) in zip(ddpm_saliency, ddim_saliency):
    corr = np.corrcoef(sal_d.flatten(), sal_i.flatten())[0, 1]
    correlations.append(corr)

fig, axes = plt.subplots(3, len(DDPM_STEPS),
                          figsize=(4*len(DDPM_STEPS), 12))
for col, ((t_d, sal_d), (t_i, sal_i), corr) in enumerate(
        zip(ddpm_saliency, ddim_saliency, correlations)):
    axes[0,col].imshow(sal_d, cmap='hot',
                        interpolation='nearest', vmin=0, vmax=1)
    axes[0,col].set_title(f"DDPM t={t_d}", fontsize=10)
    axes[0,col].axis('off')
    axes[1,col].imshow(sal_i, cmap='hot',
                        interpolation='nearest', vmin=0, vmax=1)
    axes[1,col].set_title(f"DDIM t~{t_i}", fontsize=10)
    axes[1,col].axis('off')
    diff_map = np.abs(sal_d - sal_i)
    im = axes[2,col].imshow(diff_map, cmap='coolwarm',
                             interpolation='nearest', vmin=0, vmax=0.5)
    axes[2,col].set_title(f"|DDPM-DDIM|\nr={corr:.3f}", fontsize=10)
    axes[2,col].axis('off')
    plt.colorbar(im, ax=axes[2,col], shrink=0.8)

mean_corr = np.mean(correlations)
plt.suptitle(
    f"Experiment 4: DDPM vs DDIM | Mean r={mean_corr:.3f} | Unfrozen backbone",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("exp4_ddpm_vs_ddim.png", dpi=150, bbox_inches='tight')
plt.show()

fig2, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    [f"t={t}" for t, _ in ddpm_saliency], correlations,
    color=['#2ecc71' if c > 0.7 else '#e74c3c' for c in correlations]
)
ax.axhline(mean_corr, color='black', linestyle='--',
           label=f'Mean r = {mean_corr:.3f}')
ax.axhline(0.7, color='gray', linestyle=':',
           label='r=0.7 (good agreement)')
ax.set_ylim(0, 1)
ax.set_ylabel("Pearson correlation")
ax.set_title("DDPM vs DDIM saliency correlation at matched noise levels")
ax.legend()
plt.tight_layout()
plt.savefig("exp4_correlation_barchart.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"  Mean DDPM-DDIM correlation : {mean_corr:.4f}")
print(f"  Per-timestep               : "
      f"{[f't={t}:{c:.2f}' for (t,_),c in zip(ddpm_saliency,correlations)]}")
print("  Saved: exp4_ddpm_vs_ddim.png, exp4_correlation_barchart.png")

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 52)
print("  Saliency experiment summary")
print("=" * 52)
print(f"  Exp 1: cross-timestep    — see exp1_cross_timestep.png")
print(f"  Exp 2: gate vs saliency  — Mean IoU = {mean_iou:.3f}")
print(f"  Exp 3: per-layer entropy — see exp3_per_layer.png")
print(f"  Exp 4: DDPM vs DDIM      — Mean r = {mean_corr:.3f}")
print("=" * 52)

# restore train mode
dit.train()
for pp in all_parasites:
    pp.train()

## Cell 14: Save Checkpoint

Saves everything needed to resume training or run evaluation on RunPod.

### What is saved vs frozen notebook

| Component | Frozen notebook | This notebook |
|---|---|---|
| Gate weights | ✓ | ✓ |
| Backbone weights | ✗ (unchanged) | ✓ (updated during training) |
| Optimizer states | Gate only | Backbone + gate (both groups) |
| Hyperparameters | ✓ | ✓ |
| Training history | ✓ | ✓ |
| Saliency results | ✗ | ✓ |

### File size
- Frozen checkpoint: ~15MB (gate weights only)
- Unfrozen checkpoint: ~3.2GB (full DiT backbone + gate + optimizer states)

### Storage on RunPod
Save to the persistent network volume, not the pod's ephemeral storage.
Pod storage is wiped when the pod stops. Volume persists indefinitely.

```python
SAVE_PATH = '/workspace/checkpoints/dit_sdt_unfrozen.pt'  # RunPod volume
```

In [ ]:
from google.colab import drive
import os

# ── Mount storage ──────────────────────────────────────────────────────────────
# Colab: mount Google Drive
# RunPod: use persistent volume path directly — no mounting needed

IS_RUNPOD = os.path.exists('/workspace')   # auto-detect environment

if IS_RUNPOD:
    SAVE_DIR  = '/workspace/checkpoints'
    print("RunPod environment detected.")
else:
    drive.mount('/content/drive')
    SAVE_DIR  = '/content/drive/MyDrive/dit_pruning_checkpoints'
    print("Colab environment detected.")

os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_FILENAME = 'dit_sdt_unfrozen.pt'
SAVE_PATH     = os.path.join(SAVE_DIR, SAVE_FILENAME)

# ── Collect gate state dicts ───────────────────────────────────────────────────
sdt_states = {
    f'layer_{idx}': dit.blocks[idx].predictor.state_dict()
    for idx in PRUNING_LAYERS
}

# ── Build checkpoint ───────────────────────────────────────────────────────────
checkpoint = {
    # ── model weights ──────────────────────────────────────────────────────────
    'backbone_state_dict' : dit.state_dict(),   # full DiT — includes gate blocks
    'sdt_routers'         : sdt_states,         # gate weights separately for convenience

    # ── architecture config ────────────────────────────────────────────────────
    'pruning_layers'      : PRUNING_LAYERS,
    'hidden_dim'          : hidden_dim,
    'bottleneck'          : BOTTLENECK,
    'in_channels'         : IN_CHANNELS,

    # ── training hyperparameters ───────────────────────────────────────────────
    'target_ratio'        : TARGET_RATIO,
    'lambda_dit'          : LAMBDA_DIT,
    'lambda_sparse'       : LAMBDA_SPARSE,
    'lambda_reg'          : LAMBDA_REG,
    'steps_trained'       : STEPS,
    'batch_size'          : BATCH_SIZE,
    'lr_backbone'         : LR_BACKBONE,
    'lr_gate'             : LR_GATE,
    'temp_start'          : TEMP_START,
    'temp_end'            : TEMP_END,
    'warmup_steps'        : WARMUP_STEPS,

    # ── optimizer state (both parameter groups) ────────────────────────────────
    'optimizer_state'     : optimizer.state_dict(),

    # ── training history ───────────────────────────────────────────────────────
    'history'             : history,

    # ── saliency results ───────────────────────────────────────────────────────
    'saliency'            : {
        'mean_iou'        : float(mean_iou),
        'iou_per_layer'   : [float(s) for s in iou_scores],
        'mean_ddpm_ddim_r': float(mean_corr),
        'correlations'    : [float(c) for c in correlations],
        'pruning_layers'  : PRUNING_LAYERS,
    },

    # ── environment metadata ───────────────────────────────────────────────────
    'environment'         : 'runpod' if IS_RUNPOD else 'colab',
    'dry_run'             : DRY_RUN,
    'torch_version'       : torch.__version__,
}

# ── Save ───────────────────────────────────────────────────────────────────────
print(f"\nSaving checkpoint to: {SAVE_PATH}")
print("This may take 30-60 seconds for the full unfrozen checkpoint...")

torch.save(checkpoint, SAVE_PATH)

# ── Verify ─────────────────────────────────────────────────────────────────────
file_size_gb = os.path.getsize(SAVE_PATH) / 1e9

print("\n" + "=" * 52)
print(f"  Checkpoint saved successfully")
print(f"  Path       : {SAVE_PATH}")
print(f"  Size       : {file_size_gb:.2f} GB")
print(f"  Steps      : {STEPS}")
print(f"  Dry run    : {DRY_RUN}")
print(f"  Mean IoU   : {mean_iou:.4f}")
print(f"  Mean r     : {mean_corr:.4f}")
print(f"  Keep ratio : {history['keep_ratio'][-1]*100:.1f}%")
print("=" * 52)

if DRY_RUN:
    print(f"\n  [DRY RUN] This checkpoint has only {STEPS} steps.")
    print(f"  Re-run with DRY_RUN=False on RunPod for the real checkpoint.")

## Cell 15: Load Checkpoint

Loads a saved checkpoint and restores the full training state.
Run this instead of Cells 4-13 when resuming a crashed RunPod session
or continuing training from a previous checkpoint.

### Load order matters
1. Peel any existing wrappers — prevent double-wrapping
2. Rewrap with fresh PruningWrapper instances
3. Load full backbone state dict — restores both backbone and gate positions
4. Load gate weights separately — convenience, already included in backbone
5. Restore both optimizer parameter groups
6. Restore all hyperparameters and history

### RunPod resume workflow

In [ ]:
from google.colab import drive
import os

# ── Auto-detect environment ────────────────────────────────────────────────────
IS_RUNPOD = os.path.exists('/workspace')

if IS_RUNPOD:
    LOAD_PATH = '/workspace/checkpoints/dit_sdt_unfrozen.pt'
    print("RunPod environment detected.")
else:
    drive.mount('/content/drive')
    LOAD_PATH = '/content/drive/MyDrive/dit_pruning_checkpoints/dit_sdt_unfrozen.pt'
    print("Colab environment detected.")

print(f"Loading from: {LOAD_PATH}")
checkpoint = torch.load(LOAD_PATH, map_location=device)

# ── Restore hyperparameters ────────────────────────────────────────────────────
PRUNING_LAYERS = checkpoint['pruning_layers']
hidden_dim     = checkpoint['hidden_dim']
BOTTLENECK     = checkpoint['bottleneck']
IN_CHANNELS    = checkpoint['in_channels']
TARGET_RATIO   = checkpoint['target_ratio']
LAMBDA_DIT     = checkpoint['lambda_dit']
LAMBDA_SPARSE  = checkpoint['lambda_sparse']
LAMBDA_REG     = checkpoint['lambda_reg']
STEPS          = checkpoint['steps_trained']
BATCH_SIZE     = checkpoint['batch_size']
LR_BACKBONE    = checkpoint['lr_backbone']
LR_GATE        = checkpoint['lr_gate']
TEMP_START     = checkpoint['temp_start']
TEMP_END       = checkpoint['temp_end']
WARMUP_STEPS   = checkpoint['warmup_steps']
history        = checkpoint['history']
DRY_RUN        = checkpoint['dry_run']

print(f"\nHyperparameters restored:")
print(f"  PRUNING_LAYERS : {PRUNING_LAYERS}")
print(f"  STEPS          : {STEPS}")
print(f"  BATCH_SIZE     : {BATCH_SIZE}")
print(f"  LR_BACKBONE    : {LR_BACKBONE}")
print(f"  LR_GATE        : {LR_GATE}")
print(f"  DRY_RUN        : {DRY_RUN}")

# ── Step 1: peel existing wrappers ─────────────────────────────────────────────
print("\nStep 1: Peeling existing wrappers...")
for idx in PRUNING_LAYERS:
    block = dit.blocks[idx]
    depth = 0
    while isinstance(block, PruningWrapper):
        block = block.block
        depth += 1
    dit.blocks[idx] = block
    if depth > 0:
        print(f"  Layer {idx:2d}: peeled {depth} wrapper(s)")

# ── Step 2: rewrap with fresh PruningWrappers ──────────────────────────────────
print("\nStep 2: Rewrapping layers...")
for idx in PRUNING_LAYERS:
    predictor       = TokenPredictor(hidden_dim, bottleneck=BOTTLENECK).to(device)
    dit.blocks[idx] = PruningWrapper(dit.blocks[idx], predictor)
print(f"  {len(PRUNING_LAYERS)} layers wrapped.")

# ── Step 3: load full backbone state dict ──────────────────────────────────────
print("\nStep 3: Loading backbone state dict...")
dit.load_state_dict(checkpoint['backbone_state_dict'])
print("  Backbone + gate weights restored.")

# ── Step 4: restore convenience references ─────────────────────────────────────
all_parasites = [
    b.predictor for b in dit.blocks
    if isinstance(b, PruningWrapper)
]
print(f"\nStep 4: {len(all_parasites)} gate references restored.")

# ── Step 5: rebuild optimizer with same groups ─────────────────────────────────
print("\nStep 5: Rebuilding optimizer...")
gate_param_ids = {
    id(p)
    for pp in all_parasites
    for p in pp.parameters()
}
backbone_params_list = [
    p for p in dit.parameters()
    if id(p) not in gate_param_ids and p.requires_grad
]
gate_params_list = [
    p for pp in all_parasites
    for p in pp.parameters()
]

optimizer = torch.optim.Adam([
    {'params': backbone_params_list, 'lr': LR_BACKBONE, 'name': 'backbone'},
    {'params': gate_params_list,     'lr': LR_GATE,     'name': 'gate'},
])
optimizer.load_state_dict(checkpoint['optimizer_state'])
print("  Optimizer state restored.")

# ── Step 6: restore saliency results if available ──────────────────────────────
saliency_results = checkpoint.get('saliency', None)
if saliency_results:
    mean_iou    = saliency_results['mean_iou']
    iou_scores  = saliency_results['iou_per_layer']
    mean_corr   = saliency_results['mean_ddpm_ddim_r']
    correlations = saliency_results['correlations']
    print(f"\nSaliency results restored:")
    print(f"  Mean IoU   : {mean_iou:.4f}")
    print(f"  Mean r     : {mean_corr:.4f}")
else:
    mean_iou     = None
    iou_scores   = None
    mean_corr    = None
    correlations = None
    print("\n  No saliency results in checkpoint.")

# ── Step 7: re-register hooks ──────────────────────────────────────────────────
print("\nStep 7: Re-registering hooks...")
hook_manager._registered = False   # allow re-registration after rewrap
hook_manager._handles    = []      # clear old handles
hook_manager.register_all(dit)

# ── Step 8: verify ─────────────────────────────────────────────────────────────
print("\n" + "=" * 52)
print("  Checkpoint loaded successfully")
print(f"  Path           : {LOAD_PATH}")
print(f"  Steps trained  : {STEPS}")
print(f"  History length : {len(history['loss_dit'])} entries")
print(f"  Depth check    : "
      f"blocks[0]={type(dit.blocks[0]).__name__}, "
      f"blocks[0].block={type(dit.blocks[0].block).__name__}")
print(f"  Backbone grad  : {dit.blocks[1].parameters().__next__().requires_grad}")
print("=" * 52)

if DRY_RUN:
    print(f"\n  Loaded a DRY RUN checkpoint ({STEPS} steps).")
    print(f"  Set DRY_RUN=False and adjust STEPS for full RunPod training.")